# Chapter 13: Healthcare and Scientific Agents

**Book:** *30 Agents Every AI Engineer Must Build*
**Author:** Imran Ahmad
**Publisher:** Packt Publishing

---

This notebook implements the two agent architectures from Chapter 13:

1. **Healthcare Intelligence Agent** (Sections 13.1–13.4) — Clinical
   decision support with Bayesian belief updating, FHIR data
   normalization, multi-specialist diagnostic coordination, and
   audience-adapted explanations.

2. **Scientific Discovery Agent** (Sections 13.5–13.8) — Research
   acceleration with asynchronous literature scanning, knowledge-gap
   detection, abductive hypothesis generation, and closed-loop
   experiment tracking.

**Simulation Mode:** This notebook runs fully without any API keys.
All LLM calls and external service dependencies are backed by
context-aware mock responses derived from the chapter content.

## Section 0: Setup & Configuration

In [ ]:
# =============================================================================
# Cell 0.2 — Imports
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: Technical Requirements (p.2)
# =============================================================================

import os
import sys
import json
import asyncio
import functools
import warnings
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any, Callable, Tuple

# Suppress noisy warnings before any heavy imports
warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# --- Core scientific computing ---
import numpy as np

# --- Environment management ---
from dotenv import load_dotenv
import getpass

# --- Async support for Jupyter ---
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass  # nest_asyncio is optional; needed only for async cells in Jupyter

# --- Conditional imports (graceful degradation) ---
_OPTIONAL_DEPS = {}

try:
    from scipy import stats as scipy_stats
    _OPTIONAL_DEPS["scipy"] = True
except ImportError:
    _OPTIONAL_DEPS["scipy"] = False

try:
    import aiohttp
    _OPTIONAL_DEPS["aiohttp"] = True
except ImportError:
    _OPTIONAL_DEPS["aiohttp"] = False

try:
    import fhir.resources
    _OPTIONAL_DEPS["fhir.resources"] = True
except ImportError:
    _OPTIONAL_DEPS["fhir.resources"] = False

print("=" * 60)
print("Chapter 13: Healthcare and Scientific Agents")
print("Author: Imran Ahmad")
print("=" * 60)
print(f"Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"NumPy  {np.__version__}")
for dep, available in _OPTIONAL_DEPS.items():
    status = "available" if available else "not installed (mock will be used)"
    print(f"  {dep}: {status}")
print("=" * 60)

In [ ]:
# =============================================================================
# Cell 0.4 — Color-Coded Logging Infrastructure
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: Cross-cutting (Resilience Layer Specification)
# =============================================================================

class ColorLog:
    """ANSI color codes for terminal/notebook output."""
    BLUE  = '\033[94m'
    GREEN = '\033[92m'
    RED   = '\033[91m'
    BOLD  = '\033[1m'
    RESET = '\033[0m'

# Detect ANSI color support (see troubleshooting.md Issue 7)
_SUPPORTS_COLOR = hasattr(sys.stdout, 'isatty') or 'ipykernel' in sys.modules


def log_info(message: str) -> None:
    """Log informational messages in BLUE. Used for status updates and
    simulation-mode indicators throughout the notebook."""
    if _SUPPORTS_COLOR:
        print(f"{ColorLog.BLUE}{ColorLog.BOLD}[INFO]{ColorLog.RESET} "
              f"{ColorLog.BLUE}{message}{ColorLog.RESET}")
    else:
        print(f"[INFO] {message}")


def log_success(message: str) -> None:
    """Log success messages in GREEN. Used when a step completes
    without errors."""
    if _SUPPORTS_COLOR:
        print(f"{ColorLog.GREEN}{ColorLog.BOLD}[SUCCESS]{ColorLog.RESET} "
              f"{ColorLog.GREEN}{message}{ColorLog.RESET}")
    else:
        print(f"[SUCCESS] {message}")


def log_error(message: str) -> None:
    """Log handled errors in RED. Used by @graceful_fallback when an
    exception is caught and a fallback value is returned."""
    if _SUPPORTS_COLOR:
        print(f"{ColorLog.RED}{ColorLog.BOLD}[HANDLED ERROR]{ColorLog.RESET} "
              f"{ColorLog.RED}{message}{ColorLog.RESET}")
    else:
        print(f"[HANDLED ERROR] {message}")


# --- Quick verification ---
log_info("Color-coded logging initialized.")
log_success("Green channel operational.")
log_error("Red channel operational (this is a test, not a real error).")

In [ ]:
# =============================================================================
# Cell 0.3 — API Key Management (Zero-Hardcode Policy)
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: Cross-cutting
#
# IMPORTANT: No API keys are ever hardcoded. The notebook loads keys from
# the .env file, falls back to interactive prompt, and ultimately activates
# Simulation Mode if no key is provided. See .env.template for details.
# =============================================================================

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    log_info("No OPENAI_API_KEY found in environment. Prompting...")
    try:
        OPENAI_API_KEY = getpass.getpass(
            "Enter your OpenAI API key (or press Enter for Simulation Mode): "
        )
    except EOFError:
        # Non-interactive environment (e.g., CI/CD)
        OPENAI_API_KEY = ""

SIMULATION_MODE = not bool(OPENAI_API_KEY)

if SIMULATION_MODE:
    log_info(
        "SIMULATION MODE ACTIVE — All LLM calls use context-aware "
        "mock responses derived from Chapter 13 content. "
        "No API keys required."
    )
else:
    log_success(
        f"Live API mode active. Key loaded (ends ...{OPENAI_API_KEY[-4:]})."
    )

# Load optional literature API keys (used by ProductionLiteratureScanner)
PUBMED_API_KEY = os.getenv("PUBMED_API_KEY", "")
SCOPUS_API_KEY = os.getenv("SCOPUS_API_KEY", "")
IEEE_API_KEY = os.getenv("IEEE_API_KEY", "")

## Section 1: Simulation Infrastructure

This section builds the complete simulation layer that enables the notebook
to run without any external dependencies. Every mock class and data constant
is derived directly from Chapter 13 content.

**Components:**
- `MockResponse` dataclass — standardized response envelope
- `MockLLM` — context-aware mock LLM with 6 response domains
- 9 mock data constants — patient vitals, diagnoses, FHIR bundles, papers, etc.
- ~30 mock stub classes — drop-in replacements for all architecture dependencies
- `@graceful_fallback` decorator — resilience wrapper for all I/O methods

In [ ]:
# =============================================================================
# Cell 1.1 — MockResponse Dataclass
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: Cross-cutting (Simulation Mode Specification)
# =============================================================================

@dataclass
class MockResponse:
    """Standardized response envelope for all mock LLM calls.

    Mirrors the interface of LangChain message objects so that
    downstream agent code can consume mock responses without
    conditional branching.
    """
    content: str
    metadata: Dict[str, Any] = field(default_factory=dict)

    def __str__(self) -> str:
        return self.content

log_success("MockResponse dataclass defined.")

In [ ]:
# =============================================================================
# Cell 1.2 — MockLLM: Context-Aware Mock Language Model
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.1, §13.3, §13.5–§13.7
# =============================================================================

class MockLLM:
    """Context-aware mock LLM returning domain-appropriate responses.

    The response registry contains 6 context types, each mapping to a
    clinically or scientifically plausible response derived from the
    chapter text. This ensures that Simulation Mode output is both
    meaningful and traceable to specific book sections.

    Chapter 13: Healthcare and Scientific Agents
    Book: 30 Agents Every AI Engineer Must Build
    Author: Imran Ahmad
    """

    def __init__(self):
        self.call_count = 0
        self._response_registry = {
            "diagnostic": {
                "response": (
                    "Based on presented symptoms (fever 38.9C, tachycardia "
                    "118 bpm, elevated WBC 18.4), differential diagnosis "
                    "suggests: 1) Urosepsis (p=0.61), 2) Pneumonia-source "
                    "sepsis (p=0.21), 3) Biliary source (p=0.11)."
                ),
                "section": "13.3 Clinical Decision Support"
            },
            "drug_interaction": {
                "response": (
                    "WARNING: Concurrent use of warfarin and aspirin "
                    "increases bleeding risk. Recommend INR monitoring "
                    "every 48 hours per AHA guideline v2024.2."
                ),
                "section": "13.1 Medical Knowledge Integration"
            },
            "patient_summary": {
                "response": (
                    "Your temperature, heart rate, and blood test results "
                    "together suggest your body may be fighting a serious "
                    "infection. Your care team has been notified."
                ),
                "section": "13.3 Audience-Adapted Explanation"
            },
            "literature_synthesis": {
                "response": (
                    "Cluster analysis of 12,347 papers reveals 47 thematic "
                    "groups. Dominant clusters: aromatic polyimide synthesis "
                    "(n=2,841), nanocomposite reinforcement (n=1,923), "
                    "processing-property relationships (n=1,456)."
                ),
                "section": "13.5 Literature Synthesis"
            },
            "knowledge_gap": {
                "response": (
                    "Gap identified: block copolymer architectures with "
                    "thermally stable aromatic monomers. P(referenced)=0.73, "
                    "P(directly_studied)=0.04. Novelty: 0.89, Feasibility: 0.71."
                ),
                "section": "13.6 Knowledge Gap Identification"
            },
            "hypothesis": {
                "response": (
                    "Hypothesis H1: Alternating aromatic dianhydride-diamine "
                    "block copolymer with segment length 15-20 repeat units "
                    "will achieve Tg > 350C with elongation at break > 15%. "
                    "Testability score: 0.82."
                ),
                "section": "13.7 Hypothesis Generation"
            },
        }

    def invoke(self, prompt: str, context_type: str = "diagnostic") -> MockResponse:
        """Return a domain-appropriate mock response.

        Args:
            prompt: The input prompt (logged but not parsed in mock mode).
            context_type: One of the 6 registry keys that determines
                which pre-built response to return.

        Returns:
            MockResponse with content and traceability metadata.
        """
        self.call_count += 1
        entry = self._response_registry.get(
            context_type, self._response_registry["diagnostic"]
        )
        log_info(
            f"[SIMULATION] MockLLM call #{self.call_count} | "
            f"context: {context_type} | source: {entry['section']}"
        )
        return MockResponse(
            content=entry["response"],
            metadata={"simulation": True, "section": entry["section"]}
        )

    def get_call_count(self) -> int:
        """Return the total number of mock invocations."""
        return self.call_count


# --- Initialize the global LLM reference ---
if SIMULATION_MODE:
    llm = MockLLM()
    log_success("MockLLM initialized with 6 context-type response registry.")
else:
    # In live mode, this would be replaced with a real LLM client.
    # Example: from langchain_openai import ChatOpenAI
    #          llm = ChatOpenAI(api_key=OPENAI_API_KEY, model="gpt-4o")
    llm = MockLLM()  # Fallback safety net
    log_info("Live LLM client would be initialized here. Using MockLLM as safety net.")

In [ ]:
# =============================================================================
# Cell 1.3 — Mock Data Constants (9 Datasets)
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.1–§13.8
#
# Every dataset includes a _source field tracing it to the chapter section.
# =============================================================================

# ── Dataset 1: Patient Vitals (§13.1, §13.3) ──────────────────────────────
MOCK_PATIENT_VITALS = {
    "patient_id": "SIM-PT-00421",
    "temperature_c": 38.9,
    "heart_rate_bpm": 118,
    "blood_pressure_systolic": 92,
    "blood_pressure_diastolic": 58,
    "wbc_count": 18.4,
    "wbc_left_shift": True,
    "spo2_percent": 94,
    "map_mmhg": 65,
    "lactate_mmol": 2.8,
    "_source": "13.3 Sepsis scenario from clinician report"
}

# ── Dataset 2: Diagnoses and Prior Belief (§13.1) ─────────────────────────
MOCK_DIAGNOSES = [
    "urosepsis", "pneumonia_sepsis", "biliary_sepsis",
    "viral_syndrome", "dehydration"
]
MOCK_PRIOR_BELIEF = np.array([0.25, 0.25, 0.15, 0.20, 0.15])

# ── Dataset 3: Likelihood Model (§13.1) ───────────────────────────────────
MOCK_LIKELIHOOD_SCORES = {
    "urosepsis":        0.82,
    "pneumonia_sepsis":  0.65,
    "biliary_sepsis":    0.45,
    "viral_syndrome":    0.20,
    "dehydration":       0.15,
}

# ── Dataset 4: Drug Interaction Records (§13.1) ──────────────────────────
MOCK_DRUG_DB = [
    {
        "drug_pair": ("warfarin", "aspirin"),
        "severity": "HIGH",
        "mechanism": "Additive anticoagulant effect",
        "recommendation": "Monitor INR every 48h",
        "source": "DrugBank v5.1.11",
        "guideline": "AHA 2024.2",
        "provenance": {
            "source": "drugbank",
            "version": "5.1.11",
            "retrieved_at": "2026-03-15T08:00:00Z",
            "confidence": 0.97,
        },
    },
    {
        "drug_pair": ("metformin", "contrast_dye"),
        "severity": "MODERATE",
        "mechanism": "Risk of lactic acidosis",
        "recommendation": "Hold metformin 48h pre/post contrast",
        "source": "FDA Label 2025-revision",
        "guideline": "ACR Manual on Contrast Media 2025",
        "provenance": {
            "source": "fda_labels",
            "version": "2025.1",
            "retrieved_at": "2026-03-15T08:00:00Z",
            "confidence": 0.95,
        },
    },
]

# ── Dataset 5: FHIR Bundle (§13.2) ───────────────────────────────────────
MOCK_FHIR_BUNDLE = {
    "resourceType": "Bundle",
    "type": "collection",
    "entry": [
        {
            "resource": {
                "resourceType": "Patient",
                "id": "SIM-PT-00421",
                "name": [{"family": "Simulation", "given": ["Test"]}],
                "gender": "male",
                "birthDate": "1958-07-15",
            }
        },
        {
            "resource": {
                "resourceType": "Observation",
                "id": "obs-temp-001",
                "code": {
                    "coding": [{
                        "system": "http://loinc.org",
                        "code": "8310-5",
                        "display": "Body temperature",
                    }]
                },
                "valueQuantity": {"value": 38.9, "unit": "Cel"},
                "effectiveDateTime": "2026-03-30T10:30:00Z",
            }
        },
        {
            "resource": {
                "resourceType": "Observation",
                "id": "obs-hr-001",
                "code": {
                    "coding": [{
                        "system": "http://loinc.org",
                        "code": "8867-4",
                        "display": "Heart rate",
                    }]
                },
                "valueQuantity": {"value": 118, "unit": "/min"},
                "effectiveDateTime": "2026-03-30T10:30:00Z",
            }
        },
        {
            "resource": {
                "resourceType": "Condition",
                "id": "cond-dm2-001",
                "code": {
                    "coding": [{
                        "system": "http://snomed.info/sct",
                        "code": "44054006",
                        "display": "Type 2 diabetes mellitus",
                    }]
                },
                "clinicalStatus": {"coding": [{"code": "active"}]},
                "onsetDateTime": "2018-04-01",
            }
        },
    ],
}

# ── Dataset 6: Deployment Metrics (§13.4) ─────────────────────────────────
MOCK_DEPLOYMENT_METRICS = {
    "patient_population": 200_000,
    "provider_sites": 20,
    "early_detection_improvement_pct": 30,
    "false_alarm_rate_pct": 3,
    "clinician_response_improvement_pct": 40,
    "physician_satisfaction_pct": 92,
    "differential_privacy_epsilon": 1.0,
    "raw_data_rate_kb_per_sec": 4,
    "derived_feature_size_bytes": 200,
    "transmission_interval_min": 15,
    "_source": "13.4 Diagnostic Assistance Case Study",
}

# ── Dataset 7: Paper Corpus (§13.5) ──────────────────────────────────────
MOCK_PAPER_CORPUS = [
    {
        "doi": "10.1234/sim-polymer-001",
        "title": "Thermal Stability of Aromatic Polyimides: A Comprehensive Review",
        "authors": ["Zhang, L.", "Kumar, R."],
        "journal": "Progress in Polymer Science",
        "year": 2024,
        "abstract": "This review covers recent advances in aromatic polyimide synthesis.",
        "citations": 187,
        "cluster": "aromatic_polyimide_synthesis",
        "source_db": "scopus",
    },
    {
        "doi": "10.1234/sim-polymer-002",
        "title": "Nanocomposite Reinforcement Strategies for High-Temperature Polymers",
        "authors": ["Park, S.", "Chen, W.", "Ahmed, F."],
        "journal": "Composites Science and Technology",
        "year": 2024,
        "abstract": "Nanocomposite reinforcement techniques for polyimide matrices.",
        "citations": 142,
        "cluster": "nanocomposite_reinforcement",
        "source_db": "pubmed",
    },
    {
        "doi": "10.1234/sim-polymer-003",
        "title": "Processing-Property Relationships in Semi-Crystalline Polyimides",
        "authors": ["Nakamura, T.", "Hoffman, D."],
        "journal": "Polymer",
        "year": 2023,
        "abstract": "Study of how processing conditions affect mechanical properties.",
        "citations": 98,
        "cluster": "processing_property",
        "source_db": "scopus",
    },
    {
        "doi": "10.1234/sim-polymer-004",
        "title": "Block Copolymer Architectures for Mechanical Toughening",
        "authors": ["Kim, J.", "Williams, A."],
        "journal": "Macromolecules",
        "year": 2025,
        "abstract": "Novel block copolymer designs that enhance fracture toughness.",
        "citations": 63,
        "cluster": "block_copolymer_mechanical",
        "source_db": "arxiv",
    },
    {
        "doi": "10.1234/sim-polymer-005",
        "title": "High-Temperature Homopolymer Design via Computational Screening",
        "authors": ["Patel, R.", "Liu, X.", "Okonkwo, C."],
        "journal": "ACS Applied Polymer Materials",
        "year": 2025,
        "abstract": "Computational screening of monomers for Tg > 400C homopolymers.",
        "citations": 41,
        "cluster": "high_temp_homopolymer",
        "source_db": "ieee",
    },
    {
        "doi": "10.1234/sim-polymer-006",
        "title": "Aromatic Dianhydride Selection for Aerospace Polyimide Films",
        "authors": ["Zhang, L.", "Singh, P."],
        "journal": "Journal of Applied Polymer Science",
        "year": 2024,
        "abstract": "Systematic study of dianhydride monomers for aerospace applications.",
        "citations": 156,
        "cluster": "aromatic_polyimide_synthesis",
        "source_db": "scopus",
    },
    {
        "doi": "10.1234/sim-polymer-007",
        "title": "Carbon Nanotube Dispersion in Polyimide Matrices",
        "authors": ["Lee, H.", "Martinez, G."],
        "journal": "Carbon",
        "year": 2023,
        "abstract": "Optimizing CNT dispersion for improved thermal-mechanical properties.",
        "citations": 211,
        "cluster": "nanocomposite_reinforcement",
        "source_db": "pubmed",
    },
    {
        "doi": "10.1234/sim-polymer-008",
        "title": "Imidization Kinetics and Film Formation in Polyamic Acid Systems",
        "authors": ["Brown, T.", "Nakamura, T."],
        "journal": "Polymer Engineering & Science",
        "year": 2024,
        "abstract": "Kinetic modeling of the imidization process during film casting.",
        "citations": 77,
        "cluster": "processing_property",
        "source_db": "scopus",
    },
    {
        "doi": "10.1234/sim-polymer-009",
        "title": "Segmented Block Copolyimides with Tunable Flexibility",
        "authors": ["Williams, A.", "Park, S."],
        "journal": "Polymer Chemistry",
        "year": 2025,
        "abstract": "Tuning hard/soft segment ratios for controlled flexibility.",
        "citations": 34,
        "cluster": "block_copolymer_mechanical",
        "source_db": "arxiv",
    },
    {
        "doi": "10.1234/sim-polymer-010",
        "title": "Fluorinated Polyimides for Low-Dielectric Aerospace Applications",
        "authors": ["Gupta, M.", "O'Brien, K."],
        "journal": "High Performance Polymers",
        "year": 2024,
        "abstract": "Fluorinated monomers reduce dielectric constant while maintaining Tg.",
        "citations": 89,
        "cluster": "high_temp_homopolymer",
        "source_db": "ieee",
    },
    {
        "doi": "10.1234/sim-polymer-011",
        "title": "Graphene Oxide Intercalation in Layered Polyimide Nanocomposites",
        "authors": ["Chen, W.", "Rossi, L."],
        "journal": "ACS Nano",
        "year": 2025,
        "abstract": "Graphene oxide sheets improve barrier properties of polyimide films.",
        "citations": 128,
        "cluster": "nanocomposite_reinforcement",
        "source_db": "scopus",
    },
    {
        "doi": "10.1234/sim-polymer-012",
        "title": "Solvent Effects on Polyimide Chain Packing and Tg",
        "authors": ["Hoffman, D.", "Tanaka, Y."],
        "journal": "Journal of Polymer Science",
        "year": 2023,
        "abstract": "How casting solvent choice influences chain packing density.",
        "citations": 65,
        "cluster": "processing_property",
        "source_db": "pubmed",
    },
    {
        "doi": "10.1234/sim-polymer-013",
        "title": "Thermoplastic Polyimide Blends for 3D Printing Applications",
        "authors": ["Ahmed, F.", "Kim, J."],
        "journal": "Additive Manufacturing",
        "year": 2025,
        "abstract": "Blend design enabling FDM printing of high-Tg polyimide parts.",
        "citations": 52,
        "cluster": "processing_property",
        "source_db": "arxiv",
    },
    {
        "doi": "10.1234/sim-polymer-014",
        "title": "Bio-Based Diamines for Sustainable Aromatic Polyimides",
        "authors": ["Okonkwo, C.", "Larsen, E."],
        "journal": "Green Chemistry",
        "year": 2024,
        "abstract": "Renewable diamine monomers that maintain thermal stability.",
        "citations": 93,
        "cluster": "aromatic_polyimide_synthesis",
        "source_db": "scopus",
    },
    {
        "doi": "10.1234/sim-polymer-015",
        "title": "Multi-Scale Modeling of Polyimide Mechanical Response",
        "authors": ["Liu, X.", "Brown, T."],
        "journal": "Computational Materials Science",
        "year": 2025,
        "abstract": "Molecular dynamics to continuum bridging for property prediction.",
        "citations": 47,
        "cluster": "block_copolymer_mechanical",
        "source_db": "ieee",
    },
    {
        "doi": "10.1234/sim-polymer-016",
        "title": "Polyimide Aerogels: Synthesis, Properties, and Aerospace Uses",
        "authors": ["Singh, P.", "Martinez, G.", "Patel, R."],
        "journal": "Chemistry of Materials",
        "year": 2024,
        "abstract": "Ultra-lightweight polyimide aerogels for thermal insulation.",
        "citations": 174,
        "cluster": "high_temp_homopolymer",
        "source_db": "scopus",
    },
]

# ── Dataset 8: Gap Report (§13.6) ────────────────────────────────────────
MOCK_GAP_REPORT = {
    "gaps": [
        {
            "id": "GAP-001",
            "description": (
                "Block copolymer architectures with thermally "
                "stable aromatic monomers for aerospace applications"
            ),
            "strategy": "cross_domain_intersection",
            "p_referenced": 0.73,
            "p_directly_studied": 0.04,
            "novelty_score": 0.89,
            "feasibility_score": 0.71,
            "impact_score": 0.85,
            "domain_a": "block_copolymer_mechanical",
            "domain_b": "high_temp_homopolymer",
        },
        {
            "id": "GAP-002",
            "description": (
                "Humidity-dependent degradation kinetics of "
                "UV-exposed polymer coatings"
            ),
            "strategy": "negative_space",
            "p_referenced": 0.68,
            "p_directly_studied": 0.09,
            "novelty_score": 0.74,
            "feasibility_score": 0.82,
            "impact_score": 0.63,
            "domain_a": "polymer_aging",
            "domain_b": None,
        },
        {
            "id": "GAP-003",
            "description": (
                "Long-term creep behavior of nanocomposite-"
                "reinforced polyimides under cyclic thermal loading"
            ),
            "strategy": "temporal_trend",
            "p_referenced": 0.55,
            "p_directly_studied": 0.12,
            "novelty_score": 0.66,
            "feasibility_score": 0.58,
            "impact_score": 0.72,
            "domain_a": "nanocomposite_reinforcement",
            "domain_b": "processing_property",
        },
    ],
    "methodology_used": ["negative_space", "cross_domain", "temporal_trend"],
    "_source": "13.6 Knowledge Gap Identification",
}

# ── Dataset 9: Experiment Rounds (§13.8) ─────────────────────────────────
MOCK_EXPERIMENT_ROUNDS = [
    {
        "round": 1,
        "hypothesis_id": "H1-aromatic-block",
        "predicted": {"tg_celsius": 338, "tensile_mpa": 95, "elongation_pct": 13.2},
        "measured":  {"tg_celsius": 355, "tensile_mpa": 108, "elongation_pct": 15.8},
        "avg_error_pct": 12.0,
    },
    {
        "round": 2,
        "hypothesis_id": "H1-aromatic-block-v2",
        "predicted": {"tg_celsius": 348, "tensile_mpa": 102, "elongation_pct": 15.1},
        "measured":  {"tg_celsius": 352, "tensile_mpa": 107, "elongation_pct": 16.0},
        "avg_error_pct": 8.0,
    },
    {
        "round": 3,
        "hypothesis_id": "H1-aromatic-block-v3",
        "predicted": {"tg_celsius": 353, "tensile_mpa": 106, "elongation_pct": 15.7},
        "measured":  {"tg_celsius": 355, "tensile_mpa": 108, "elongation_pct": 15.8},
        "avg_error_pct": 5.0,
    },
]

# ── Additional mock structures for fallback targets ──────────────────────

# Used by DiagnosticCoordinator @graceful_fallback (§13.3)
MOCK_DIAGNOSTIC_REPORT = {
    "patient_id": "SIM-PT-00421",
    "differential": [
        {"diagnosis": "urosepsis", "probability": 0.61, "rank": 1},
        {"diagnosis": "pneumonia_sepsis", "probability": 0.21, "rank": 2},
        {"diagnosis": "biliary_sepsis", "probability": 0.11, "rank": 3},
    ],
    "confidence": 0.85,
    "safety_escalation": True,
    "escalation_reason": "MAP < 70 mmHg — possible septic shock",
    "_source": "13.3 Diagnostic Coordinator fallback",
}

# Used by PatientDataPipeline @graceful_fallback (§13.2)
MOCK_CLINICAL_CONTEXT = {
    "patient_id": "SIM-PT-00421",
    "vitals": MOCK_PATIENT_VITALS,
    "conditions": ["Type 2 diabetes mellitus"],
    "temporal_alignment": "synchronized",
    "data_quality": "complete",
    "_source": "13.2 Patient Data Pipeline fallback",
}

# Used by HypothesisGenerator @graceful_fallback (§13.7)
MOCK_HYPOTHESES = {
    "hypotheses": [
        {
            "id": "H1",
            "statement": (
                "Alternating aromatic dianhydride-diamine block copolymer "
                "with segment length 15-20 repeat units will achieve "
                "Tg > 350C with elongation at break > 15%."
            ),
            "reasoning_type": "abductive",
            "consistency_score": 0.78,
            "testability_score": 0.82,
            "proposed_experiments": [
                "Synthesize BPDA-PDA / BPDA-ODA alternating blocks",
                "DMA sweep 25-500C at 1 Hz",
                "Tensile test at 25C per ASTM D882",
            ],
        },
        {
            "id": "H2",
            "statement": (
                "Incorporating 5 wt% boron nitride nanosheets into the "
                "polyimide matrix will increase thermal conductivity by "
                "40% without reducing Tg below 300C."
            ),
            "reasoning_type": "analogical",
            "consistency_score": 0.71,
            "testability_score": 0.88,
            "proposed_experiments": [
                "Solution blending BN nanosheets at 1, 3, 5, 7 wt%",
                "Laser flash analysis for thermal diffusivity",
                "DSC for Tg measurement",
            ],
        },
    ],
    "_source": "13.7 Hypothesis Generation",
}

log_success(f"9 mock datasets loaded. Total paper corpus size: {len(MOCK_PAPER_CORPUS)}.")

In [ ]:
# =============================================================================
# Cell 1.4 — Mock Stub Classes (Architecture Dependencies)
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.1–§13.8
#
# Each stub:
#   - Accepts the same constructor args as the book code
#   - Returns domain-appropriate mock data from the registry
#   - Logs via log_info with [SIMULATION] prefix and section reference
#   - Never raises exceptions (all errors caught internally)
# =============================================================================


# ─────────────────────────────────────────────────────────────────────────────
# §13.1 — Medical Knowledge Integration Stubs
# ─────────────────────────────────────────────────────────────────────────────

class DrugInteractionDB:
    """Mock drug interaction database (§13.1).
    Author: Imran Ahmad"""
    def __init__(self, db_path: str = "", version: str = "5.1.11"):
        self.db_path = db_path
        self.version = version

    def query(self, medications: List[str]) -> List[dict]:
        log_info(f"[SIMULATION] DrugInteractionDB.query({medications}) — §13.1")
        return MOCK_DRUG_DB

    def check_pair(self, drug_a: str, drug_b: str) -> Optional[dict]:
        log_info(f"[SIMULATION] DrugInteractionDB.check_pair({drug_a}, {drug_b}) — §13.1")
        for record in MOCK_DRUG_DB:
            if set(record["drug_pair"]) == {drug_a, drug_b}:
                return record
        return None


class ClinicalGuidelineEngine:
    """Mock clinical guideline engine (§13.1).
    Author: Imran Ahmad"""
    def __init__(self, guideline_sources: Optional[List[str]] = None):
        self.guideline_sources = guideline_sources or ["AHA", "IDSA", "WHO"]

    def lookup(self, condition: str) -> dict:
        log_info(f"[SIMULATION] ClinicalGuidelineEngine.lookup('{condition}') — §13.1")
        return {
            "condition": condition,
            "guidelines": [
                {"source": "IDSA 2024", "recommendation": "Blood cultures x2 before antibiotics"},
                {"source": "Surviving Sepsis 2024", "recommendation": "Broad-spectrum abx within 1h"},
            ],
            "_source": "13.1 Clinical Guideline Engine mock",
        }


class DiseaseOntology:
    """Mock disease ontology for ICD/SNOMED lookups (§13.1).
    Author: Imran Ahmad"""
    def __init__(self, ontology_version: str = "2025.1"):
        self.ontology_version = ontology_version

    def resolve(self, term: str) -> dict:
        log_info(f"[SIMULATION] DiseaseOntology.resolve('{term}') — §13.1")
        return {
            "term": term,
            "icd10": "A41.9",
            "snomed": "91302008",
            "display": "Sepsis, unspecified organism",
            "_source": "13.1 Disease Ontology mock",
        }


class MedicalLiteratureIndex:
    """Mock medical literature search index (§13.1).
    Author: Imran Ahmad"""
    def __init__(self, index_name: str = "pubmed"):
        self.index_name = index_name

    def search(self, query: str, max_results: int = 5) -> List[dict]:
        log_info(f"[SIMULATION] MedicalLiteratureIndex.search('{query}') — §13.1")
        return [
            {"pmid": "SIM-38291", "title": "Sepsis biomarker validation study",
             "relevance": 0.92},
            {"pmid": "SIM-38292", "title": "Urinary tract infection prognosis",
             "relevance": 0.87},
        ]


# ─────────────────────────────────────────────────────────────────────────────
# §13.2 — FHIR Normalization & Data Pipeline Stubs
# ─────────────────────────────────────────────────────────────────────────────

class FHIRResourceValidator:
    """Mock FHIR resource validator (§13.2).
    Author: Imran Ahmad"""
    def validate(self, resource: dict) -> dict:
        rtype = resource.get("resourceType", "Unknown")
        log_info(f"[SIMULATION] FHIRResourceValidator.validate('{rtype}') — §13.2")
        return {"valid": True, "resourceType": rtype, "errors": []}


class HL7v2ToFHIRAdapter:
    """Mock HL7v2-to-FHIR conversion adapter (§13.2).
    Author: Imran Ahmad"""
    def convert(self, hl7_message: str) -> dict:
        log_info("[SIMULATION] HL7v2ToFHIRAdapter.convert() — §13.2")
        return MOCK_FHIR_BUNDLE["entry"][0]["resource"]


class PassthroughAdapter:
    """Passthrough adapter for already-FHIR-compliant data (§13.2).
    Author: Imran Ahmad"""
    def convert(self, resource: dict) -> dict:
        log_info("[SIMULATION] PassthroughAdapter.convert() — §13.2")
        return resource


class CSVLabResultAdapter:
    """Mock CSV lab result to FHIR converter (§13.2).
    Author: Imran Ahmad"""
    def convert(self, csv_row: dict) -> dict:
        log_info("[SIMULATION] CSVLabResultAdapter.convert() — §13.2")
        return {
            "resourceType": "Observation",
            "code": {"text": csv_row.get("test_name", "Unknown")},
            "valueQuantity": {"value": csv_row.get("value", 0)},
        }


class EpicFHIRAdapter:
    """Mock Epic EHR FHIR adapter (§13.2).
    Author: Imran Ahmad"""
    def __init__(self, endpoint: str = ""):
        self.endpoint = endpoint

    def fetch_patient(self, patient_id: str) -> dict:
        log_info(f"[SIMULATION] EpicFHIRAdapter.fetch_patient('{patient_id}') — §13.2")
        return MOCK_FHIR_BUNDLE


class CernerFHIRAdapter:
    """Mock Cerner EHR FHIR adapter (§13.2).
    Author: Imran Ahmad"""
    def __init__(self, endpoint: str = ""):
        self.endpoint = endpoint

    def fetch_patient(self, patient_id: str) -> dict:
        log_info(f"[SIMULATION] CernerFHIRAdapter.fetch_patient('{patient_id}') — §13.2")
        return MOCK_FHIR_BUNDLE


# ─────────────────────────────────────────────────────────────────────────────
# §13.3 — Diagnostic Coordinator Sub-Agent Stubs
# ─────────────────────────────────────────────────────────────────────────────

class BiometricAnalyzer:
    """Mock biometric data analyzer (§13.3).
    Author: Imran Ahmad"""
    def analyze(self, vitals: dict) -> dict:
        log_info("[SIMULATION] BiometricAnalyzer.analyze() — §13.3")
        return {
            "anomalies": ["tachycardia", "hypotension", "leukocytosis"],
            "severity": "HIGH",
            "sepsis_screen_positive": True,
        }


class SymptomInterpreter:
    """Mock symptom interpretation engine (§13.3).
    Author: Imran Ahmad"""
    def interpret(self, symptoms: List[str]) -> dict:
        log_info(f"[SIMULATION] SymptomInterpreter.interpret({symptoms}) — §13.3")
        return {
            "primary_pattern": "systemic inflammatory response",
            "associated_conditions": ["sepsis", "SIRS"],
            "urgency": "IMMEDIATE",
        }


class PatientHistoryAgent:
    """Mock patient history retrieval agent (§13.3).
    Author: Imran Ahmad"""
    def retrieve(self, patient_id: str) -> dict:
        log_info(f"[SIMULATION] PatientHistoryAgent.retrieve('{patient_id}') — §13.3")
        return {
            "patient_id": patient_id,
            "history": [
                {"condition": "Type 2 diabetes mellitus", "onset": "2018-04-01"},
                {"condition": "Hypertension", "onset": "2015-09-15"},
            ],
            "allergies": ["sulfonamides"],
            "current_medications": ["metformin", "lisinopril"],
        }


class DifferentialGenerator:
    """Mock differential diagnosis generator (§13.3).
    Author: Imran Ahmad"""
    def generate(self, clinical_context: dict) -> List[dict]:
        log_info("[SIMULATION] DifferentialGenerator.generate() — §13.3")
        return [
            {"diagnosis": "urosepsis", "probability": 0.61, "rank": 1},
            {"diagnosis": "pneumonia_sepsis", "probability": 0.21, "rank": 2},
            {"diagnosis": "biliary_sepsis", "probability": 0.11, "rank": 3},
            {"diagnosis": "viral_syndrome", "probability": 0.05, "rank": 4},
            {"diagnosis": "dehydration", "probability": 0.02, "rank": 5},
        ]


class ClinicalExplainer:
    """Mock explanation generator for clinicians and patients (§13.3).
    Author: Imran Ahmad"""
    def explain_for_clinician(self, diagnosis_report: dict) -> str:
        log_info("[SIMULATION] ClinicalExplainer.explain_for_clinician() — §13.3")
        return (
            "CLINICAL SUMMARY: Patient SIM-PT-00421 presents with fever "
            "(38.9C), tachycardia (118 bpm), hypotension (MAP 65), and "
            "leukocytosis (WBC 18.4 with left shift). Lactate 2.8 mmol/L "
            "suggests tissue hypoperfusion. Primary differential: urosepsis "
            "(p=0.61). Recommend: blood cultures x2, broad-spectrum "
            "antibiotics within 1 hour, 30 mL/kg crystalloid bolus."
        )

    def explain_for_patient(self, diagnosis_report: dict) -> str:
        log_info("[SIMULATION] ClinicalExplainer.explain_for_patient() — §13.3")
        return (
            "Your temperature, heart rate, and blood test results together "
            "suggest your body may be fighting a serious infection. Your "
            "care team has been notified and will start treatment right "
            "away. They will give you fluids through an IV and antibiotics "
            "to help fight the infection."
        )


class ConfidenceAwareAgent:
    """Mock confidence-calibrated decision agent (§13.3).
    Author: Imran Ahmad"""
    def __init__(self, escalation_threshold: float = 0.15):
        self.escalation_threshold = escalation_threshold

    def assess(self, differential: List[dict], vitals: dict) -> dict:
        log_info("[SIMULATION] ConfidenceAwareAgent.assess() — §13.3")
        top_prob = differential[0]["probability"] if differential else 0
        needs_escalation = vitals.get("map_mmhg", 999) < 70
        return {
            "overall_confidence": 0.85,
            "calibration_method": "bayesian_posterior",
            "escalation_triggered": needs_escalation,
            "escalation_reason": (
                "MAP < 70 mmHg — possible septic shock" if needs_escalation
                else "No escalation criteria met"
            ),
        }


class ClinicalMemorySystem:
    """Mock clinical memory for longitudinal patient tracking (§13.3).
    Author: Imran Ahmad"""
    def __init__(self):
        self._store = {}

    def record(self, patient_id: str, event: dict) -> None:
        log_info(f"[SIMULATION] ClinicalMemorySystem.record('{patient_id}') — §13.3")
        self._store.setdefault(patient_id, []).append(event)

    def recall(self, patient_id: str) -> List[dict]:
        log_info(f"[SIMULATION] ClinicalMemorySystem.recall('{patient_id}') — §13.3")
        return self._store.get(patient_id, [])


class SafetyMonitor:
    """Mock safety monitor for clinical decision oversight (§13.3).
    Author: Imran Ahmad"""
    def check(self, action: str, context: dict) -> dict:
        log_info(f"[SIMULATION] SafetyMonitor.check('{action}') — §13.3")
        return {
            "action": action,
            "approved": True,
            "warnings": ["Ensure human clinician reviews before acting"],
            "regulatory_note": "Educational simulation only — not for clinical use",
        }


class AuditTrailGenerator:
    """Mock audit trail logger for regulatory compliance (§13.3).
    Author: Imran Ahmad"""
    def __init__(self):
        self.trail = []

    def log_event(self, event_type: str, details: dict) -> None:
        log_info(f"[SIMULATION] AuditTrailGenerator.log_event('{event_type}') — §13.3")
        entry = {
            "timestamp": datetime.now().isoformat(),
            "event_type": event_type,
            "details": details,
        }
        self.trail.append(entry)

    def get_trail(self) -> List[dict]:
        return self.trail


# ─────────────────────────────────────────────────────────────────────────────
# §13.5 — Literature Scanner Stubs
# ─────────────────────────────────────────────────────────────────────────────

class PubMedClient:
    """Mock PubMed API client (§13.5).
    Author: Imran Ahmad"""
    def __init__(self, api_key: str = ""):
        self.api_key = api_key

    async def search(self, query: str, max_results: int = 50) -> List[dict]:
        log_info(f"[SIMULATION] PubMedClient.search('{query}') — §13.5")
        return [p for p in MOCK_PAPER_CORPUS if p["source_db"] == "pubmed"]


class ArxivClient:
    """Mock arXiv API client (§13.5).
    Author: Imran Ahmad"""
    async def search(self, query: str, max_results: int = 50) -> List[dict]:
        log_info(f"[SIMULATION] ArxivClient.search('{query}') — §13.5")
        return [p for p in MOCK_PAPER_CORPUS if p["source_db"] == "arxiv"]


class ScopusClient:
    """Mock Scopus API client (§13.5).
    Author: Imran Ahmad"""
    def __init__(self, api_key: str = ""):
        self.api_key = api_key

    async def search(self, query: str, max_results: int = 50) -> List[dict]:
        log_info(f"[SIMULATION] ScopusClient.search('{query}') — §13.5")
        return [p for p in MOCK_PAPER_CORPUS if p["source_db"] == "scopus"]


class IEEEXploreClient:
    """Mock IEEE Xplore API client (§13.5).
    Author: Imran Ahmad"""
    def __init__(self, api_key: str = ""):
        self.api_key = api_key

    async def search(self, query: str, max_results: int = 50) -> List[dict]:
        log_info(f"[SIMULATION] IEEEXploreClient.search('{query}') — §13.5")
        return [p for p in MOCK_PAPER_CORPUS if p["source_db"] == "ieee"]


class ResultCache:
    """Mock result cache for literature queries (§13.5).
    Author: Imran Ahmad"""
    def __init__(self):
        self._cache = {}
        self.hits = 0
        self.misses = 0

    def get(self, key: str) -> Optional[List[dict]]:
        if key in self._cache:
            self.hits += 1
            log_info(f"[SIMULATION] ResultCache HIT for '{key}' — §13.5")
            return self._cache[key]
        self.misses += 1
        return None

    def put(self, key: str, value: List[dict]) -> None:
        self._cache[key] = value

    def stats(self) -> dict:
        return {"hits": self.hits, "misses": self.misses, "size": len(self._cache)}


class PaperDeduplicator:
    """Mock paper deduplication engine (§13.5).
    Author: Imran Ahmad"""
    def deduplicate(self, papers: List[dict]) -> Tuple[List[dict], int]:
        seen_dois = set()
        unique = []
        dupes = 0
        for paper in papers:
            doi = paper.get("doi", "")
            if doi not in seen_dois:
                seen_dois.add(doi)
                unique.append(paper)
            else:
                dupes += 1
        log_info(f"[SIMULATION] PaperDeduplicator: {len(unique)} unique, {dupes} duplicates — §13.5")
        return unique, dupes


# ─────────────────────────────────────────────────────────────────────────────
# §13.6 — Knowledge Gap Detection Stubs
# ─────────────────────────────────────────────────────────────────────────────

class CitationGraphAnalyzer:
    """Mock citation graph analyzer (§13.6).
    Author: Imran Ahmad"""
    def analyze(self, papers: List[dict]) -> dict:
        log_info("[SIMULATION] CitationGraphAnalyzer.analyze() — §13.6")
        return {
            "total_nodes": len(papers),
            "total_edges": len(papers) * 3,
            "clusters": 5,
            "isolated_nodes": 2,
        }


class ResearchDomainMapper:
    """Mock research domain mapper for thematic clustering (§13.6).
    Author: Imran Ahmad"""
    def map_domains(self, papers: List[dict]) -> dict:
        log_info("[SIMULATION] ResearchDomainMapper.map_domains() — §13.6")
        clusters = {}
        for paper in papers:
            cluster = paper.get("cluster", "unknown")
            clusters.setdefault(cluster, []).append(paper["doi"])
        return {
            "cluster_count": len(clusters),
            "clusters": {k: len(v) for k, v in clusters.items()},
        }


class TemporalTrendTracker:
    """Mock temporal trend tracker for publication analysis (§13.6).
    Author: Imran Ahmad"""
    def track(self, papers: List[dict]) -> dict:
        log_info("[SIMULATION] TemporalTrendTracker.track() — §13.6")
        by_year = {}
        for paper in papers:
            yr = paper.get("year", 0)
            by_year[yr] = by_year.get(yr, 0) + 1
        return {
            "year_distribution": by_year,
            "trend": "increasing" if len(by_year) > 1 else "stable",
        }


# ─────────────────────────────────────────────────────────────────────────────
# §13.7 — Hypothesis Generation Stubs
# ─────────────────────────────────────────────────────────────────────────────

class TheoreticalFrameworkRetriever:
    """Mock theoretical framework retriever (§13.7).
    Author: Imran Ahmad"""
    def retrieve(self, domain: str) -> dict:
        log_info(f"[SIMULATION] TheoreticalFrameworkRetriever.retrieve('{domain}') — §13.7")
        return {
            "domain": domain,
            "frameworks": [
                "Structure-property relationships in block copolymers",
                "Flory-Huggins theory for polymer miscibility",
                "Time-temperature superposition principle",
            ],
        }


class ScientificReasoningEngine:
    """Mock scientific reasoning engine for abductive inference (§13.7).
    Author: Imran Ahmad"""
    def reason(self, gap: dict, frameworks: dict) -> dict:
        log_info("[SIMULATION] ScientificReasoningEngine.reason() — §13.7")
        return {
            "reasoning_type": "abductive",
            "conclusion": (
                "If aromatic dianhydride blocks provide thermal stability "
                "and flexible diamine blocks provide elongation, then an "
                "alternating architecture should achieve both properties."
            ),
            "confidence": 0.78,
        }


class HypothesisEvaluator:
    """Mock hypothesis evaluation and scoring engine (§13.7).
    Author: Imran Ahmad"""
    def evaluate(self, hypothesis: dict) -> dict:
        log_info("[SIMULATION] HypothesisEvaluator.evaluate() — §13.7")
        return {
            "hypothesis_id": hypothesis.get("id", "H?"),
            "consistency_score": 0.78,
            "testability_score": 0.82,
            "novelty_score": 0.85,
            "overall_score": 0.82,
        }


# ─────────────────────────────────────────────────────────────────────────────
# §13.8 — Experiment Tracking Stubs
# ─────────────────────────────────────────────────────────────────────────────

class PredictionDatabase:
    """Mock prediction storage for experiment tracking (§13.8).
    Author: Imran Ahmad"""
    def __init__(self):
        self._predictions = []

    def store(self, prediction: dict) -> None:
        log_info("[SIMULATION] PredictionDatabase.store() — §13.8")
        self._predictions.append(prediction)

    def get_all(self) -> List[dict]:
        return self._predictions


class ExperimentalResultDatabase:
    """Mock experimental result storage (§13.8).
    Author: Imran Ahmad"""
    def __init__(self):
        self._results = []

    def store(self, result: dict) -> None:
        log_info("[SIMULATION] ExperimentalResultDatabase.store() — §13.8")
        self._results.append(result)

    def get_all(self) -> List[dict]:
        return self._results


class FeedbackEngine:
    """Mock feedback engine for closed-loop learning (§13.8).
    Author: Imran Ahmad"""
    def compute_feedback(self, predicted: dict, measured: dict) -> dict:
        log_info("[SIMULATION] FeedbackEngine.compute_feedback() — §13.8")
        errors = {}
        for key in predicted:
            if key in measured and isinstance(predicted[key], (int, float)):
                pred_val = predicted[key]
                meas_val = measured[key]
                if meas_val != 0:
                    errors[key] = abs(pred_val - meas_val) / abs(meas_val) * 100
        avg_error = np.mean(list(errors.values())) if errors else 0.0
        return {
            "per_property_error_pct": errors,
            "avg_error_pct": round(avg_error, 1),
            "recommendation": "Adjust model parameters" if avg_error > 5 else "Model converged",
        }


log_success(f"All mock stub classes defined — {30} architecture dependency stubs ready.")

In [ ]:
# =============================================================================
# Cell 1.5 — @graceful_fallback Decorator
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: Cross-cutting (Resilience Layer Specification)
#
# This decorator wraps every agent method that performs I/O, inference,
# or external access. It catches exceptions, logs them in RED, and
# returns a fallback value — ensuring the notebook never crashes.
# =============================================================================

def graceful_fallback(fallback_value=None, section_ref: str = "Unknown"):
    """Resilience decorator that catches exceptions and returns a fallback.

    Ensures the notebook never terminates on a single tool failure.
    All caught errors are logged via log_error (red) with the section
    reference for traceability.

    Args:
        fallback_value: A static value or callable that produces the
            fallback. If callable, it is invoked with no arguments
            when the decorated function raises an exception.
        section_ref: The book section reference (e.g., "13.1") for
            traceability in log output.

    Usage:
        @graceful_fallback(
            fallback_value=lambda: MOCK_DRUG_DB,
            section_ref="13.1 Drug Interaction Check"
        )
        def check_drug_interactions(medications):
            return drug_db_client.query(medications)
    """
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            try:
                result = func(*args, **kwargs)
                log_success(
                    f"Step complete. {func.__name__} returned valid output."
                )
                return result
            except Exception as e:
                log_error(
                    f"{type(e).__name__}: {e}. "
                    f"Falling back to mock logic for §{section_ref}."
                )
                if callable(fallback_value):
                    return fallback_value()
                return fallback_value
        return wrapper
    return decorator


# ── Verification: test the decorator with a deliberate error ─────────────

@graceful_fallback(fallback_value=lambda: {"status": "fallback_activated"}, section_ref="TEST")
def _test_graceful_fallback():
    """Deliberately raises an error to verify the decorator."""
    raise ConnectionError("Simulated external service failure")

test_result = _test_graceful_fallback()
assert test_result == {"status": "fallback_activated"}, "Decorator test failed!"
log_success(f"@graceful_fallback decorator verified. Test result: {test_result}")
print()
log_success(
    "=== Section 1 Complete === "
    "Simulation infrastructure fully operational. "
    "All mock classes, data constants, and resilience decorators are ready."
)

## Section 2: Healthcare Intelligence Agent (§13.1–§13.4)

This section implements the clinical decision-support pipeline from the book:

- **2a:** Bayesian Belief Update — the mathematical foundation (§13.1)
- **2b:** ClinicalKnowledgeBase — multi-source medical knowledge with provenance (§13.1)
- **2c:** FHIR Normalization + Patient Data Pipeline — standards-compliant data ingestion (§13.2)
- **2d:** DiagnosticCoordinator — end-to-end multi-specialist pipeline (§13.3)
- **2e:** Case Study Metrics — deployment outcomes visualization (§13.4)

> **Safety Note:** This is educational code. All patient data is synthetic.
> The 0.15 escalation threshold is illustrative, not clinically validated.
> See Section 13.9 of the book for production deployment considerations.

In [ ]:
# =============================================================================
# Cell 2a — Bayesian Belief Update
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.1 (p.3) — Medical Knowledge Integration
#
# This is the ONE fully executable function — it runs live with NumPy,
# not through the mock layer. It demonstrates Bayes' theorem:
#   posterior ∝ likelihood × prior
# =============================================================================

def update_belief(
    prior: np.ndarray,
    likelihood_scores: Dict[str, float],
    diagnoses: List[str],
) -> np.ndarray:
    """Bayesian belief update for diagnostic probability estimation.

    Implements the core formula from Section 13.1:
        posterior[i] = likelihood[i] * prior[i] / sum(likelihood * prior)

    This is the mathematical foundation upon which the Healthcare
    Intelligence Agent builds its diagnostic confidence model.

    Args:
        prior: Array of prior probabilities for each diagnosis.
            Must sum to 1.0 and have the same length as diagnoses.
        likelihood_scores: Dict mapping diagnosis names to their
            likelihood given the observed evidence (vitals, labs, etc.).
        diagnoses: List of diagnosis names, in the same order as prior.

    Returns:
        Normalized posterior probability array.

    Chapter 13: Healthcare and Scientific Agents
    Book: 30 Agents Every AI Engineer Must Build
    Author: Imran Ahmad
    """
    # Build likelihood vector in the same order as diagnoses
    likelihood = np.array([
        likelihood_scores.get(dx, 0.01) for dx in diagnoses
    ])

    # Bayes' theorem: posterior proportional to likelihood times prior
    unnormalized = likelihood * prior

    # Normalize to ensure valid probability distribution
    total = unnormalized.sum()
    if total == 0:
        log_error("Zero total in Bayesian update — returning uniform prior. §13.1")
        return np.ones_like(prior) / len(prior)

    posterior = unnormalized / total
    return posterior


# -- Demo: Run Bayesian belief update with mock clinical data --

log_info("Running Bayesian Belief Update (§13.1, p.3)...")
print()

posterior = update_belief(
    prior=MOCK_PRIOR_BELIEF,
    likelihood_scores=MOCK_LIKELIHOOD_SCORES,
    diagnoses=MOCK_DIAGNOSES,
)

# Display results
print("+----------------------+----------+---------+----------------+")
print("| Bayesian Belief Update Results (§13.1)                     |")
print("+----------------------+----------+---------+----------------+")
print("| Diagnosis            | Prior    | L(D|E)  | Posterior      |")
print("+----------------------+----------+---------+----------------+")
for i, dx in enumerate(MOCK_DIAGNOSES):
    lk = MOCK_LIKELIHOOD_SCORES.get(dx, 0.01)
    print(f"| {dx:<20s} | {MOCK_PRIOR_BELIEF[i]:.3f}    | {lk:.3f}   | {posterior[i]:.4f}         |")
print("+----------------------+----------+---------+----------------+")
print(f"| {'TOTAL':<20s} | {MOCK_PRIOR_BELIEF.sum():.3f}    |         | {posterior.sum():.4f}         |")
print("+----------------------+----------+---------+----------------+")
print()

# Highlight the posterior shift
top_idx = np.argmax(posterior)
log_success(
    f"Posterior shift: {MOCK_DIAGNOSES[top_idx]} rose from "
    f"{MOCK_PRIOR_BELIEF[top_idx]:.2f} -> {posterior[top_idx]:.4f} "
    f"(highest posterior probability)."
)

In [ ]:
# =============================================================================
# Cell 2b — ClinicalKnowledgeBase
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.1 (pp.5-7) — Medical Knowledge Integration
# =============================================================================

class ClinicalKnowledgeBase:
    """Multi-source clinical knowledge integration with provenance tracking.

    Integrates:
      - Drug interaction databases (DrugBank)
      - Clinical practice guidelines (AHA, IDSA, WHO)
      - Disease ontologies (ICD-10, SNOMED CT)
      - Medical literature indices (PubMed)

    Each query result carries provenance metadata so the clinician
    can trace back to the original evidence source.

    Chapter 13: Healthcare and Scientific Agents
    Book: 30 Agents Every AI Engineer Must Build
    Author: Imran Ahmad
    """

    def __init__(
        self,
        drug_db=None,
        guideline_engine=None,
        disease_ontology=None,
        literature_index=None,
    ):
        self.drug_db = drug_db or DrugInteractionDB()
        self.guideline_engine = guideline_engine or ClinicalGuidelineEngine()
        self.disease_ontology = disease_ontology or DiseaseOntology()
        self.literature_index = literature_index or MedicalLiteratureIndex()

    @graceful_fallback(
        fallback_value=lambda: MOCK_DRUG_DB,
        section_ref="13.1 Clinical Knowledge Query"
    )
    def query(self, clinical_context: dict) -> dict:
        """Query all knowledge sources for a given clinical context."""
        condition = clinical_context.get("condition", "sepsis")
        medications = clinical_context.get("medications", [])

        drug_results = self.drug_db.query(medications)
        guideline_results = self.guideline_engine.lookup(condition)
        ontology_results = self.disease_ontology.resolve(condition)
        literature_results = self.literature_index.search(condition)

        resolved = self._resolve_conflicts(
            drug_results, guideline_results, ontology_results
        )

        return {
            "condition": condition,
            "drug_interactions": drug_results,
            "guidelines": guideline_results,
            "ontology": ontology_results,
            "literature": literature_results,
            "conflict_resolution": resolved,
            "provenance": {
                "sources_queried": 4,
                "timestamp": datetime.now().isoformat(),
                "confidence": 0.93,
            },
        }

    def _resolve_conflicts(self, drug_results, guidelines, ontology):
        """Resolve conflicts between knowledge sources.
        Hierarchy from §13.1, p.7:
          1. Clinical practice guidelines (highest authority)
          2. Disease ontology (standardized definitions)
          3. Literature evidence (supporting data)
        """
        return {
            "method": "hierarchical_precedence",
            "hierarchy": [
                "clinical_guidelines",
                "disease_ontology",
                "literature_evidence",
            ],
            "conflicts_found": 0,
            "resolution_notes": "No conflicts in mock data — all sources aligned.",
        }


# -- Demo: Query clinical knowledge base --

log_info("Demonstrating ClinicalKnowledgeBase (§13.1, pp.5-7)...")
print()

ckb = ClinicalKnowledgeBase()
clinical_context = {
    "condition": "sepsis",
    "medications": ["warfarin", "aspirin", "metformin"],
    "symptoms": ["fever", "tachycardia", "hypotension"],
}

kb_result = ckb.query(clinical_context)
print()

print("+-------------------------------------------------------------+")
print("| ClinicalKnowledgeBase Query Results (§13.1)                 |")
print("+-------------------------------------------------------------+")
print(f"| Condition queried:    {kb_result.get('condition', 'N/A'):<37s}|")
print(f"| Sources queried:     {kb_result['provenance']['sources_queried']:<38d}|")
print(f"| Overall confidence:  {kb_result['provenance']['confidence']:<38.2f}|")
print(f"| Conflicts found:     {kb_result['conflict_resolution']['conflicts_found']:<38d}|")
print("+-------------------------------------------------------------+")
print("| Drug Interactions Found:                                     |")
for interaction in kb_result.get("drug_interactions", []):
    pair = " + ".join(interaction["drug_pair"])
    sev = interaction["severity"]
    print(f"|   [{sev:<8s}] {pair:<46s}|")
    print(f"|             {interaction['recommendation']:<46s}|")
print("+-------------------------------------------------------------+")
print("| Conflict Resolution Hierarchy:                               |")
for rank, source in enumerate(kb_result["conflict_resolution"]["hierarchy"], 1):
    print(f"|   {rank}. {source:<55s}|")
print("+-------------------------------------------------------------+")
print()
log_success("ClinicalKnowledgeBase demonstrated with provenance tracking.")

In [ ]:
# =============================================================================
# Cell 2c — FHIRNormalizationLayer + PatientDataPipeline
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.2 (pp.8-10) — Multi-Modal Medical Data Processing
# =============================================================================

class FHIRNormalizationLayer:
    """FHIR-compliant data normalization for heterogeneous EHR sources.

    Handles data from multiple EHR systems (Epic, Cerner, CSV labs,
    HL7v2 messages) by routing each source through the appropriate
    adapter before validation.

    Chapter 13 | Author: Imran Ahmad
    """

    def __init__(self, validator=None, adapters=None):
        self.validator = validator or FHIRResourceValidator()
        self.adapters = adapters or {
            "epic": EpicFHIRAdapter(),
            "cerner": CernerFHIRAdapter(),
            "csv_lab": CSVLabResultAdapter(),
            "hl7v2": HL7v2ToFHIRAdapter(),
            "passthrough": PassthroughAdapter(),
        }

    @graceful_fallback(
        fallback_value=lambda: [MOCK_FHIR_BUNDLE],
        section_ref="13.2 FHIR Normalization"
    )
    def normalize(self, bundle: dict) -> List[dict]:
        """Normalize a FHIR bundle into validated individual resources."""
        normalized = []
        entries = bundle.get("entry", [])
        skipped = 0

        for entry in entries:
            resource = entry.get("resource", {})
            resource_type = resource.get("resourceType", "Unknown")

            validation = self.validator.validate(resource)
            if validation.get("valid", False):
                normalized.append(resource)
            else:
                log_error(
                    f"FHIR validation failed for {resource_type}: "
                    f"{validation.get('errors', [])} — §13.2"
                )
                skipped += 1

        log_info(
            f"[SIMULATION] Normalized {len(normalized)} resources, "
            f"skipped {skipped} — §13.2"
        )
        return normalized


class PatientDataPipeline:
    """Patient data ingestion and temporal alignment pipeline.

    Processes normalized FHIR resources into a unified clinical
    context object with temporal alignment.

    Chapter 13 | Author: Imran Ahmad
    """

    def __init__(self, normalization_layer=None):
        self.normalization_layer = normalization_layer or FHIRNormalizationLayer()

    @graceful_fallback(
        fallback_value=lambda: MOCK_CLINICAL_CONTEXT,
        section_ref="13.2 Patient Data Pipeline"
    )
    def process(self, bundle: dict, patient_id: str = "") -> dict:
        """Process a FHIR bundle into a temporally aligned clinical context."""
        # Step 1: Normalize
        resources = self.normalization_layer.normalize(bundle)

        # Step 2: Extract by resource type
        patient_info = {}
        observations = []
        conditions = []

        for resource in resources:
            rtype = resource.get("resourceType", "")
            if rtype == "Patient":
                patient_info = resource
            elif rtype == "Observation":
                coding = resource.get("code", {}).get("coding", [{}])
                observations.append({
                    "id": resource.get("id", ""),
                    "code": coding[0].get("display", "Unknown") if coding else "Unknown",
                    "value": resource.get("valueQuantity", {}).get("value"),
                    "unit": resource.get("valueQuantity", {}).get("unit", ""),
                    "timestamp": resource.get("effectiveDateTime", ""),
                })
            elif rtype == "Condition":
                coding = resource.get("code", {}).get("coding", [{}])
                clin_st = resource.get("clinicalStatus", {}).get("coding", [{}])
                conditions.append({
                    "id": resource.get("id", ""),
                    "display": coding[0].get("display", "Unknown") if coding else "Unknown",
                    "status": clin_st[0].get("code", "unknown") if clin_st else "unknown",
                    "onset": resource.get("onsetDateTime", ""),
                })

        # Step 3: Temporal alignment
        timestamps = [obs["timestamp"] for obs in observations if obs["timestamp"]]
        alignment = "synchronized" if len(set(timestamps)) <= 1 else "multi-timepoint"

        pid = patient_info.get("id", patient_id or "unknown")

        clinical_context = {
            "patient_id": pid,
            "patient_info": patient_info,
            "observations": observations,
            "conditions": conditions,
            "temporal_alignment": alignment,
            "data_quality": "complete" if observations and conditions else "incomplete",
            "resource_count": len(resources),
            "_source": "13.2 Patient Data Pipeline output",
        }

        log_info(
            f"[SIMULATION] Pipeline processed {len(resources)} resources "
            f"for patient {pid}. Alignment: {alignment} — §13.2"
        )
        return clinical_context


# -- Demo: Run FHIR normalization and patient pipeline --

log_info("Demonstrating FHIRNormalizationLayer + PatientDataPipeline (§13.2, pp.8-10)...")
print()

pipeline = PatientDataPipeline()
clinical_ctx = pipeline.process(MOCK_FHIR_BUNDLE, patient_id="SIM-PT-00421")
print()

print("+-------------------------------------------------------------+")
print("| Patient Data Pipeline Output (§13.2)                        |")
print("+-------------------------------------------------------------+")
print(f"| Patient ID:          {clinical_ctx['patient_id']:<38s}|")
print(f"| Resources processed: {clinical_ctx['resource_count']:<38d}|")
print(f"| Temporal alignment:  {clinical_ctx['temporal_alignment']:<38s}|")
print(f"| Data quality:        {clinical_ctx['data_quality']:<38s}|")
print("+-------------------------------------------------------------+")
print("| Observations:                                                |")
for obs in clinical_ctx.get("observations", []):
    val_str = f"{obs['value']} {obs['unit']}" if obs['value'] is not None else "N/A"
    print(f"|   {obs['code']:<30s} = {val_str:<24s}|")
print("+-------------------------------------------------------------+")
print("| Active Conditions:                                           |")
for cond in clinical_ctx.get("conditions", []):
    print(f"|   [{cond['status']:<8s}] {cond['display']:<46s}|")
    print(f"|             Onset: {cond['onset']:<40s}|")
print("+-------------------------------------------------------------+")
print()
log_success("Patient data pipeline demonstrated with temporal alignment.")

In [ ]:
# =============================================================================
# Cell 2d — DiagnosticCoordinator: Full End-to-End Pipeline
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.3 (pp.12-15) — Clinical Decision Support Pipeline
#
# Orchestrates the 8-step diagnostic workflow:
#   1. Analyze biometrics        5. Bayesian belief update
#   2. Interpret symptoms        6. Confidence assessment
#   3. Retrieve patient history  7. Generate explanations
#   4. Generate differential     8. Safety check + audit trail
# =============================================================================

class DiagnosticCoordinator:
    """Multi-specialist diagnostic coordination pipeline.

    Orchestrates a team of sub-agents through a structured diagnostic
    workflow. Implements the 'specialist team' pattern from §13.3.

    Key design decisions (§13.3):
      - Escalation threshold triggers human clinician review
        when MAP < 70 mmHg or confidence is low (p.14)
      - Dual-audience explanations: clinician + patient (p.15)
      - Full audit trail for regulatory traceability

    Chapter 13 | Author: Imran Ahmad
    """

    ESCALATION_THRESHOLD = 0.15  # Illustrative, not clinically validated

    def __init__(
        self,
        biometric_analyzer=None,
        symptom_interpreter=None,
        history_agent=None,
        differential_gen=None,
        explainer=None,
        confidence_agent=None,
        memory_system=None,
        safety_monitor=None,
        audit_generator=None,
        llm_client=None,
    ):
        self.biometric_analyzer = biometric_analyzer or BiometricAnalyzer()
        self.symptom_interpreter = symptom_interpreter or SymptomInterpreter()
        self.history_agent = history_agent or PatientHistoryAgent()
        self.differential_gen = differential_gen or DifferentialGenerator()
        self.explainer = explainer or ClinicalExplainer()
        self.confidence_agent = confidence_agent or ConfidenceAwareAgent(
            escalation_threshold=self.ESCALATION_THRESHOLD
        )
        self.memory_system = memory_system or ClinicalMemorySystem()
        self.safety_monitor = safety_monitor or SafetyMonitor()
        self.audit_generator = audit_generator or AuditTrailGenerator()
        self.llm = llm_client or llm  # Use global llm (MockLLM or real)

    @graceful_fallback(
        fallback_value=lambda: MOCK_DIAGNOSTIC_REPORT,
        section_ref="13.3 Diagnostic Coordinator"
    )
    def diagnose(self, patient_id: str, vitals: dict) -> dict:
        """Run the full diagnostic pipeline for a patient.
        Orchestrates all sub-agents through the 8-step workflow (§13.3).
        """
        log_info(f"Starting diagnostic pipeline for patient {patient_id} — §13.3")

        # Step 1: Analyze biometrics
        biometric_results = self.biometric_analyzer.analyze(vitals)

        # Step 2: Interpret symptom patterns
        symptoms = biometric_results.get("anomalies", [])
        symptom_analysis = self.symptom_interpreter.interpret(symptoms)

        # Step 3: Retrieve patient history
        history = self.history_agent.retrieve(patient_id)

        # Step 4: Generate differential diagnosis
        clinical_context = {
            "vitals": vitals,
            "biometrics": biometric_results,
            "symptoms": symptom_analysis,
            "history": history,
        }
        differential = self.differential_gen.generate(clinical_context)

        # Step 5: Bayesian belief update to refine probabilities
        diagnosis_names = [d["diagnosis"] for d in differential]
        prior = np.array([d["probability"] for d in differential])
        prior = prior / prior.sum()  # Ensure valid distribution

        posterior = update_belief(
            prior=prior,
            likelihood_scores=MOCK_LIKELIHOOD_SCORES,
            diagnoses=diagnosis_names,
        )

        for i, dx in enumerate(differential):
            dx["posterior"] = float(round(posterior[i], 4))
        differential.sort(key=lambda x: x["posterior"], reverse=True)

        # Step 6: Confidence assessment and escalation check
        confidence_result = self.confidence_agent.assess(differential, vitals)

        # Step 7: Generate dual-audience explanations
        diagnosis_report = {
            "patient_id": patient_id,
            "differential": differential,
            "confidence": confidence_result,
        }
        clinician_explanation = self.explainer.explain_for_clinician(diagnosis_report)
        patient_explanation = self.explainer.explain_for_patient(diagnosis_report)

        # LLM-enhanced summary (mock in simulation mode)
        llm_response = self.llm.invoke(
            f"Summarize diagnostic findings for patient {patient_id}",
            context_type="diagnostic"
        )

        # Step 8: Safety check and audit trail
        safety_result = self.safety_monitor.check(
            action="diagnostic_recommendation",
            context=clinical_context,
        )

        self.audit_generator.log_event("diagnosis_started", {
            "patient_id": patient_id, "vitals_snapshot": vitals
        })
        self.audit_generator.log_event("differential_generated", {
            "count": len(differential),
            "top_diagnosis": differential[0]["diagnosis"],
        })
        self.audit_generator.log_event("escalation_decision", {
            "triggered": confidence_result["escalation_triggered"],
            "reason": confidence_result.get("escalation_reason", ""),
        })
        self.audit_generator.log_event("explanations_generated", {
            "audiences": ["clinician", "patient"],
        })

        # Record in clinical memory
        self.memory_system.record(patient_id, {
            "event": "diagnostic_run",
            "timestamp": datetime.now().isoformat(),
            "top_diagnosis": differential[0]["diagnosis"],
            "confidence": confidence_result["overall_confidence"],
        })

        return {
            "patient_id": patient_id,
            "differential": differential,
            "confidence": confidence_result,
            "explanations": {
                "clinician": clinician_explanation,
                "patient": patient_explanation,
                "llm_summary": llm_response.content,
            },
            "safety": safety_result,
            "audit_trail": self.audit_generator.get_trail(),
            "_source": "13.3 DiagnosticCoordinator.diagnose() output",
        }


# -- Demo: Full end-to-end diagnostic pipeline --

log_info("Running full DiagnosticCoordinator pipeline (§13.3, pp.12-15)...")
print()

coordinator = DiagnosticCoordinator()
report = coordinator.diagnose(
    patient_id="SIM-PT-00421",
    vitals=MOCK_PATIENT_VITALS,
)
print()

# -- Display: Differential Ranking --
print("+------+--------------------+-----------+--------------------+")
print("| DIAGNOSTIC REPORT — Patient SIM-PT-00421 (§13.3)          |")
print("+------+--------------------+-----------+--------------------+")
print("| Rank | Diagnosis          | Posterior | Prior              |")
print("+------+--------------------+-----------+--------------------+")
for i, dx in enumerate(report["differential"], 1):
    print(f"|  {i:<4d}| {dx['diagnosis']:<18s} | {dx['posterior']:<9.4f} | {dx['probability']:<18.4f} |")
print("+------+--------------------+-----------+--------------------+")
print()

# -- Display: Confidence & Escalation --
conf = report["confidence"]
esc_marker = "!! YES !!" if conf["escalation_triggered"] else "No"
print("+-------------------------------------------------------------+")
print("| Confidence Assessment (§13.3)                                |")
print("+-------------------------------------------------------------+")
print(f"| Overall confidence:  {conf['overall_confidence']:<38.2f}|")
print(f"| Calibration method:  {conf['calibration_method']:<38s}|")
print(f"| Escalation:          {esc_marker:<38s}|")
if conf["escalation_triggered"]:
    print(f"| Reason:              {conf['escalation_reason']:<38s}|")
print("+-------------------------------------------------------------+")
print()

# -- Display: Clinician Explanation (§13.3, p.15) --
print("+-------------------------------------------------------------+")
print("| CLINICIAN EXPLANATION (§13.3, p.15)                          |")
print("+-------------------------------------------------------------+")
clin_text = report["explanations"]["clinician"]
# Word-wrap at ~57 chars
words = clin_text.split()
line = ""
for w in words:
    if len(line) + len(w) + 1 > 57:
        print(f"| {line:<58s}|")
        line = w
    else:
        line = f"{line} {w}".strip()
if line:
    print(f"| {line:<58s}|")
print("+-------------------------------------------------------------+")
print()

# -- Display: Patient Explanation (§13.3, p.15) --
print("+-------------------------------------------------------------+")
print("| PATIENT EXPLANATION (§13.3, p.15)                            |")
print("+-------------------------------------------------------------+")
pat_text = report["explanations"]["patient"]
words = pat_text.split()
line = ""
for w in words:
    if len(line) + len(w) + 1 > 57:
        print(f"| {line:<58s}|")
        line = w
    else:
        line = f"{line} {w}".strip()
if line:
    print(f"| {line:<58s}|")
print("+-------------------------------------------------------------+")
print()

# -- Display: Audit Trail --
print("+-------------------------------------------------------------+")
print("| AUDIT TRAIL (§13.3 — Regulatory Compliance)                 |")
print("+-------------------------------------------------------------+")
for event in report.get("audit_trail", []):
    etype = event["event_type"]
    print(f"| [{etype:<28s}] {event['timestamp'][:19]:<26s}|")
print("+-------------------------------------------------------------+")
print()
log_success("Full diagnostic pipeline demonstrated with all 8 steps.")

In [ ]:
# =============================================================================
# Cell 2e — Case Study Metrics Visualization
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.4 (pp.16-17) — Diagnostic Assistance Case Study
# =============================================================================

log_info("Displaying Case Study Deployment Metrics (§13.4, pp.16-17)...")
print()

m = MOCK_DEPLOYMENT_METRICS

print("=" * 61)
print("  DIAGNOSTIC ASSISTANCE DEPLOYMENT CASE STUDY (§13.4)")
print("  Healthcare Intelligence Agent — Real-World Outcomes")
print("=" * 61)
print(f"  Patient Population:     {m['patient_population']:>10,d}")
print(f"  Provider Sites:         {m['provider_sites']:>10d}")
print("-" * 61)
print("  KEY PERFORMANCE METRICS")
print("-" * 61)

metrics = [
    ("Early Detection",       f"+{m['early_detection_improvement_pct']}% improvement",  m['early_detection_improvement_pct']),
    ("False Alarm Rate",      f"{m['false_alarm_rate_pct']}%",                          m['false_alarm_rate_pct']),
    ("Clinician Response",    f"+{m['clinician_response_improvement_pct']}% faster",    m['clinician_response_improvement_pct']),
    ("Physician Satisfaction", f"{m['physician_satisfaction_pct']}%",                    m['physician_satisfaction_pct']),
]

for name, label, val in metrics:
    bar_len = max(1, val // 2)
    bar = "█" * bar_len
    print(f"  {name + ':':<26s} {label}")
    print(f"    [{bar:<30s}]")

print("-" * 61)
print("  PRIVACY & BANDWIDTH (§13.4, p.17)")
print("-" * 61)
print(f"  Differential Privacy e:  {m['differential_privacy_epsilon']:.1f}")
print(f"  Raw Data Rate:           {m['raw_data_rate_kb_per_sec']} KB/sec")
print(f"  Derived Feature Size:    {m['derived_feature_size_bytes']} bytes")
print(f"  Transmission Interval:   {m['transmission_interval_min']} min")
print("=" * 61)
print()
log_success(
    "Section 2 Complete — Healthcare Intelligence Agent fully demonstrated. "
    "All components from §13.1-§13.4 implemented and verified."
)

## Section 3: Scientific Discovery Agent (§13.5–§13.8)

This section implements the research acceleration pipeline from the book:

- **3a:** ProductionLiteratureScanner — async multi-source literature retrieval (§13.5)
- **3b:** KnowledgeGapDetector — 3 gap-detection strategies (§13.6)
- **3c:** HypothesisGenerator — abductive reasoning engine (§13.7)
- **3d:** ExperimentTracker — closed-loop learning with error reduction (§13.8)
- **3e:** Materials Science Case Study — full pipeline orchestration (§13.8)

> The Scientific Discovery Agent demonstrates Level 4 autonomous behavior:
> it not only executes research tasks but learns from experimental outcomes
> to refine its own predictive models over successive rounds.

In [ ]:
# =============================================================================
# Cell 3a — ProductionLiteratureScanner
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.5 (pp.19-21) — Production-Grade Literature Scanning
#
# This class demonstrates production-grade async literature retrieval
# across multiple academic databases (PubMed, arXiv, Scopus, IEEE).
# Key patterns: async fan-out, result caching, deduplication, and
# graceful degradation when individual sources fail.
# =============================================================================

class ProductionLiteratureScanner:
    """Async multi-source literature scanner with caching and deduplication.

    Implements the production-grade scanning architecture from §13.5:
      - Parallel async queries to 4 academic databases
      - Result caching to minimize redundant API calls
      - DOI-based deduplication across sources
      - Graceful degradation: failed sources are logged, not fatal

    The scanner uses asyncio.gather() to fan out queries simultaneously,
    reducing total wall-clock time from O(n) to O(1) relative to the
    number of sources.

    Chapter 13: Healthcare and Scientific Agents
    Book: 30 Agents Every AI Engineer Must Build
    Author: Imran Ahmad
    """

    def __init__(
        self,
        pubmed_client=None,
        arxiv_client=None,
        scopus_client=None,
        ieee_client=None,
        cache=None,
        deduplicator=None,
    ):
        self.pubmed = pubmed_client or PubMedClient(api_key=PUBMED_API_KEY)
        self.arxiv = arxiv_client or ArxivClient()
        self.scopus = scopus_client or ScopusClient(api_key=SCOPUS_API_KEY)
        self.ieee = ieee_client or IEEEXploreClient(api_key=IEEE_API_KEY)
        self.cache = cache or ResultCache()
        self.deduplicator = deduplicator or PaperDeduplicator()
        self.sources = {
            "pubmed": self.pubmed,
            "arxiv": self.arxiv,
            "scopus": self.scopus,
            "ieee": self.ieee,
        }

    async def _search_source(self, name: str, client, query: str, max_results: int) -> dict:
        """Search a single source with error handling."""
        try:
            results = await client.search(query, max_results=max_results)
            return {"source": name, "status": "success", "count": len(results), "papers": results}
        except Exception as e:
            log_error(f"{name} search failed: {type(e).__name__}: {e} — §13.5")
            return {"source": name, "status": "error", "count": 0, "papers": [], "error": str(e)}

    @graceful_fallback(
        fallback_value=lambda: MOCK_PAPER_CORPUS,
        section_ref="13.5 Literature Scanner"
    )
    def search_all(self, query: str, max_per_source: int = 50) -> dict:
        """Search all academic sources in parallel with caching and dedup.

        Args:
            query: Search query string for literature retrieval.
            max_per_source: Maximum results to fetch per database.

        Returns:
            Dict with unified paper list, source statuses, cache stats,
            and deduplication metrics.
        """
        # Check cache first
        cached = self.cache.get(query)
        if cached is not None:
            log_info(f"[SIMULATION] Cache hit for '{query}' — §13.5")
            return {
                "query": query,
                "papers": cached,
                "total": len(cached),
                "from_cache": True,
                "source_status": {},
                "cache_stats": self.cache.stats(),
                "dedup_stats": {"duplicates_removed": 0},
            }

        # Async fan-out to all sources
        async def _gather():
            tasks = [
                self._search_source(name, client, query, max_per_source)
                for name, client in self.sources.items()
            ]
            return await asyncio.gather(*tasks)

        source_results = asyncio.run(_gather())

        # Aggregate results
        all_papers = []
        source_status = {}
        for result in source_results:
            source_status[result["source"]] = {
                "status": result["status"],
                "count": result["count"],
            }
            all_papers.extend(result["papers"])

        # Deduplicate
        unique_papers, dupe_count = self.deduplicator.deduplicate(all_papers)

        # Cache the result
        self.cache.put(query, unique_papers)

        log_info(
            f"[SIMULATION] Scanner complete: {len(all_papers)} raw, "
            f"{len(unique_papers)} unique, {dupe_count} duplicates — §13.5"
        )

        return {
            "query": query,
            "papers": unique_papers,
            "total": len(unique_papers),
            "from_cache": False,
            "source_status": source_status,
            "cache_stats": self.cache.stats(),
            "dedup_stats": {"duplicates_removed": dupe_count},
        }


# -- Demo: Run async literature scan --

log_info("Demonstrating ProductionLiteratureScanner (§13.5, pp.19-21)...")
print()

scanner = ProductionLiteratureScanner()
scan_result = scanner.search_all("aromatic polyimide thermal stability")
print()

# Display source status
print("=" * 61)
print("  LITERATURE SCAN RESULTS (§13.5)")
print("=" * 61)
print(f"  Query: '{scan_result['query']}'")
print(f"  Total unique papers: {scan_result['total']}")
print(f"  From cache: {scan_result['from_cache']}")
print("-" * 61)
print("  SOURCE STATUS:")
for src, info in scan_result.get("source_status", {}).items():
    status_icon = "[OK]" if info["status"] == "success" else "[!!]"
    print(f"    {status_icon} {src:<12s}  {info['count']} papers")
print("-" * 61)
print("  DEDUPLICATION:")
print(f"    Duplicates removed: {scan_result['dedup_stats']['duplicates_removed']}")
print("-" * 61)
print("  CACHE STATS:")
cs = scan_result["cache_stats"]
print(f"    Hits: {cs['hits']}  |  Misses: {cs['misses']}  |  Size: {cs['size']}")
print("-" * 61)

# Show cluster distribution
clusters = {}
for paper in scan_result["papers"]:
    c = paper.get("cluster", "unknown")
    clusters[c] = clusters.get(c, 0) + 1
print("  THEMATIC CLUSTERS:")
for cluster_name, count in sorted(clusters.items(), key=lambda x: -x[1]):
    bar = "█" * (count * 3)
    print(f"    {cluster_name:<35s} ({count:2d}) {bar}")
print("=" * 61)

# Second search to demonstrate cache hit
print()
log_info("Running second search to demonstrate cache behavior...")
scan_result_2 = scanner.search_all("aromatic polyimide thermal stability")
print()
print(f"  Second search from cache: {scan_result_2['from_cache']}")
cs2 = scan_result_2["cache_stats"]
print(f"  Cache stats — Hits: {cs2['hits']}  |  Misses: {cs2['misses']}")
print()
log_success("ProductionLiteratureScanner demonstrated with async, dedup, and caching.")

In [ ]:
# =============================================================================
# Cell 3b — KnowledgeGapDetector
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.6 (pp.23-25) — Knowledge Gap Identification
#
# Implements 3 gap detection strategies:
#   1. Negative Space — topics referenced but not directly studied
#   2. Cross-Domain Intersection — gaps at boundaries between fields
#   3. Temporal Trend — declining research activity in important areas
# =============================================================================

class KnowledgeGapDetector:
    """Multi-strategy knowledge gap detection engine.

    Uses three complementary strategies to identify research gaps
    in a corpus of scientific literature (§13.6):

    1. Negative Space: Finds topics frequently referenced in citations
       but rarely the primary subject of study (high P(referenced),
       low P(directly_studied)).

    2. Cross-Domain Intersection: Identifies unexplored combinations
       at the boundary of two well-studied domains.

    3. Temporal Trend: Detects domains where publication activity is
       declining despite sustained citation interest.

    Each gap is scored on novelty, feasibility, and impact dimensions.

    Chapter 13: Healthcare and Scientific Agents
    Book: 30 Agents Every AI Engineer Must Build
    Author: Imran Ahmad
    """

    def __init__(
        self,
        citation_analyzer=None,
        domain_mapper=None,
        trend_tracker=None,
        llm_client=None,
    ):
        self.citation_analyzer = citation_analyzer or CitationGraphAnalyzer()
        self.domain_mapper = domain_mapper or ResearchDomainMapper()
        self.trend_tracker = trend_tracker or TemporalTrendTracker()
        self.llm = llm_client or llm

    @graceful_fallback(
        fallback_value=lambda: MOCK_GAP_REPORT,
        section_ref="13.6 Knowledge Gap Detection"
    )
    def detect_gaps(self, papers: list, strategies: list = None) -> dict:
        """Detect knowledge gaps using multiple strategies.

        Args:
            papers: List of paper dicts from the literature scanner.
            strategies: Which strategies to apply. Defaults to all three.

        Returns:
            Gap report with scored gaps, methodology metadata, and
            LLM-enhanced gap descriptions.
        """
        strategies = strategies or ["negative_space", "cross_domain", "temporal_trend"]

        # Analyze the corpus structure
        citation_graph = self.citation_analyzer.analyze(papers)
        domain_map = self.domain_mapper.map_domains(papers)
        trends = self.trend_tracker.track(papers)

        # Get LLM-enhanced gap analysis
        llm_response = self.llm.invoke(
            f"Analyze knowledge gaps in corpus of {len(papers)} papers "
            f"across {domain_map['cluster_count']} thematic clusters.",
            context_type="knowledge_gap"
        )

        # In simulation mode, return the pre-built gap report
        # In live mode, this would synthesize gaps from the analyses above
        gaps = MOCK_GAP_REPORT["gaps"]

        # Score and rank gaps
        ranked_gaps = sorted(
            gaps,
            key=lambda g: (g["novelty_score"] * 0.4 +
                           g["feasibility_score"] * 0.3 +
                           g["impact_score"] * 0.3),
            reverse=True,
        )

        return {
            "gaps": ranked_gaps,
            "methodology_used": strategies,
            "corpus_stats": {
                "total_papers": len(papers),
                "citation_graph": citation_graph,
                "domain_clusters": domain_map,
                "temporal_trends": trends,
            },
            "llm_analysis": llm_response.content,
            "_source": "13.6 KnowledgeGapDetector.detect_gaps() output",
        }


# -- Demo: Detect knowledge gaps --

log_info("Demonstrating KnowledgeGapDetector (§13.6, pp.23-25)...")
print()

gap_detector = KnowledgeGapDetector()
gap_result = gap_detector.detect_gaps(MOCK_PAPER_CORPUS)
print()

print("=" * 61)
print("  KNOWLEDGE GAP ANALYSIS (§13.6)")
print("=" * 61)
print(f"  Corpus size: {gap_result['corpus_stats']['total_papers']} papers")
print(f"  Strategies:  {', '.join(gap_result['methodology_used'])}")
print(f"  Gaps found:  {len(gap_result['gaps'])}")
print("-" * 61)

for i, gap in enumerate(gap_result["gaps"], 1):
    composite = (gap["novelty_score"] * 0.4 +
                 gap["feasibility_score"] * 0.3 +
                 gap["impact_score"] * 0.3)
    print(f"  GAP #{i}: {gap['id']}")
    print(f"    Strategy:     {gap['strategy']}")
    print(f"    Description:  {gap['description'][:55]}")
    if len(gap['description']) > 55:
        print(f"                  {gap['description'][55:]}")
    print(f"    P(referenced):       {gap['p_referenced']:.2f}")
    print(f"    P(directly_studied): {gap['p_directly_studied']:.2f}")
    print(f"    Novelty:     {gap['novelty_score']:.2f}  |  "
          f"Feasibility: {gap['feasibility_score']:.2f}  |  "
          f"Impact: {gap['impact_score']:.2f}")
    print(f"    Composite:   {composite:.3f}")
    if gap.get("domain_a"):
        domains = gap["domain_a"]
        if gap.get("domain_b"):
            domains += f" x {gap['domain_b']}"
        print(f"    Domains:     {domains}")
    print()

print("-" * 61)
print("  LLM ANALYSIS SUMMARY:")
analysis_text = gap_result["llm_analysis"]
words = analysis_text.split()
line = "    "
for w in words:
    if len(line) + len(w) + 1 > 59:
        print(line)
        line = "    " + w
    else:
        line += " " + w
if line.strip():
    print(line)
print("=" * 61)
print()
log_success("KnowledgeGapDetector demonstrated with all 3 strategies.")

In [ ]:
# =============================================================================
# Cell 3c — HypothesisGenerator
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.7 (pp.26-28) — Hypothesis Generation
#
# Uses abductive reasoning to generate testable hypotheses from
# identified knowledge gaps. Each hypothesis is scored for
# consistency (alignment with known theory), testability (can it
# be experimentally verified?), and novelty.
# =============================================================================

class HypothesisGenerator:
    """Abductive reasoning engine for scientific hypothesis generation.

    Given a set of knowledge gaps, generates testable hypotheses
    using three reasoning modes (§13.7):

    1. Abductive — Infer the best explanation for observed patterns
    2. Analogical — Transfer principles from related domains
    3. Combinatorial — Combine existing findings in novel ways

    Each hypothesis includes proposed experiments for validation.

    Chapter 13: Healthcare and Scientific Agents
    Book: 30 Agents Every AI Engineer Must Build
    Author: Imran Ahmad
    """

    def __init__(
        self,
        framework_retriever=None,
        reasoning_engine=None,
        evaluator=None,
        llm_client=None,
    ):
        self.framework_retriever = framework_retriever or TheoreticalFrameworkRetriever()
        self.reasoning_engine = reasoning_engine or ScientificReasoningEngine()
        self.evaluator = evaluator or HypothesisEvaluator()
        self.llm = llm_client or llm

    @graceful_fallback(
        fallback_value=lambda: MOCK_HYPOTHESES,
        section_ref="13.7 Hypothesis Generation"
    )
    def generate_hypotheses(self, gap_report: dict) -> dict:
        """Generate testable hypotheses from knowledge gaps.

        Args:
            gap_report: Output from KnowledgeGapDetector.detect_gaps().

        Returns:
            Dict with scored hypotheses and proposed experiments.
        """
        gaps = gap_report.get("gaps", [])
        if not gaps:
            log_info("[SIMULATION] No gaps provided — using mock gaps. §13.7")
            gaps = MOCK_GAP_REPORT["gaps"]

        all_hypotheses = []

        for gap in gaps[:2]:  # Focus on top 2 gaps for demo
            # Retrieve relevant theoretical frameworks
            domain = gap.get("domain_a", "polymer_science")
            frameworks = self.framework_retriever.retrieve(domain)

            # Apply scientific reasoning
            reasoning = self.reasoning_engine.reason(gap, frameworks)

            # Get LLM-enhanced hypothesis
            llm_response = self.llm.invoke(
                f"Generate hypothesis for gap: {gap['description']}",
                context_type="hypothesis"
            )

        # In simulation mode, use pre-built hypotheses
        hypotheses = MOCK_HYPOTHESES["hypotheses"]

        # Evaluate each hypothesis
        evaluated = []
        for hyp in hypotheses:
            scores = self.evaluator.evaluate(hyp)
            evaluated.append({
                **hyp,
                "evaluation": scores,
            })

        return {
            "hypotheses": evaluated,
            "gaps_addressed": len(gaps[:2]),
            "reasoning_modes_used": ["abductive", "analogical"],
            "_source": "13.7 HypothesisGenerator.generate_hypotheses() output",
        }


# -- Demo: Generate hypotheses from gap report --

log_info("Demonstrating HypothesisGenerator (§13.7, pp.26-28)...")
print()

hypothesis_gen = HypothesisGenerator()
hyp_result = hypothesis_gen.generate_hypotheses(MOCK_GAP_REPORT)
print()

print("=" * 61)
print("  HYPOTHESIS GENERATION RESULTS (§13.7)")
print("=" * 61)
print(f"  Gaps addressed: {hyp_result['gaps_addressed']}")
print(f"  Reasoning modes: {', '.join(hyp_result['reasoning_modes_used'])}")
print(f"  Hypotheses generated: {len(hyp_result['hypotheses'])}")
print("-" * 61)

for hyp in hyp_result["hypotheses"]:
    print(f"  HYPOTHESIS {hyp['id']}:")
    print(f"    Reasoning type: {hyp['reasoning_type']}")

    # Word-wrap the statement
    stmt = hyp["statement"]
    words = stmt.split()
    line = "    "
    for w in words:
        if len(line) + len(w) + 1 > 59:
            print(line)
            line = "    " + w
        else:
            line += " " + w
    if line.strip():
        print(line)
    print()

    # Scores
    ev = hyp.get("evaluation", {})
    print(f"    Consistency:  {ev.get('consistency_score', hyp.get('consistency_score', 0)):.2f}  |  "
          f"Testability: {ev.get('testability_score', hyp.get('testability_score', 0)):.2f}  |  "
          f"Novelty: {ev.get('novelty_score', 0):.2f}")
    print(f"    Overall:      {ev.get('overall_score', 0):.2f}")
    print()

    # Proposed experiments
    print("    PROPOSED EXPERIMENTS:")
    for j, exp in enumerate(hyp.get("proposed_experiments", []), 1):
        print(f"      {j}. {exp}")
    print()

print("=" * 61)
print()
log_success("HypothesisGenerator demonstrated with abductive reasoning and scoring.")

In [ ]:
# =============================================================================
# Cell 3d — ExperimentTracker: Closed-Loop Learning
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.8 (pp.31-32) — Experiment Tracking
#
# Demonstrates Level 4 autonomous behavior: the agent learns from
# experimental outcomes to refine its own predictive models. Over
# 3 rounds, prediction error decreases from 12% -> 8% -> 5%.
# =============================================================================

class ExperimentTracker:
    """Closed-loop experiment tracking with predictive model refinement.

    Records experimental predictions and measured outcomes, computes
    prediction error, and feeds back into the model. This implements
    the Level 4 learning loop from §13.8 — the agent improves its
    accuracy over successive experimental rounds.

    Chapter 13: Healthcare and Scientific Agents
    Book: 30 Agents Every AI Engineer Must Build
    Author: Imran Ahmad
    """

    def __init__(
        self,
        prediction_db=None,
        result_db=None,
        feedback_engine=None,
    ):
        self.prediction_db = prediction_db or PredictionDatabase()
        self.result_db = result_db or ExperimentalResultDatabase()
        self.feedback_engine = feedback_engine or FeedbackEngine()
        self.rounds = []

    @graceful_fallback(
        fallback_value=lambda: {},
        section_ref="13.8 Experiment Tracker"
    )
    def record_result(self, round_data: dict) -> dict:
        """Record an experimental round and compute feedback.

        Args:
            round_data: Dict with 'round', 'hypothesis_id',
                'predicted', and 'measured' property dicts.

        Returns:
            Feedback dict with per-property errors and recommendations.
        """
        predicted = round_data.get("predicted", {})
        measured = round_data.get("measured", {})

        # Store prediction and result
        self.prediction_db.store(predicted)
        self.result_db.store(measured)

        # Compute feedback
        feedback = self.feedback_engine.compute_feedback(predicted, measured)

        round_summary = {
            "round": round_data.get("round", len(self.rounds) + 1),
            "hypothesis_id": round_data.get("hypothesis_id", "unknown"),
            "predicted": predicted,
            "measured": measured,
            "feedback": feedback,
        }
        self.rounds.append(round_summary)

        log_info(
            f"[SIMULATION] Round {round_summary['round']}: "
            f"avg error = {feedback['avg_error_pct']:.1f}% — "
            f"{feedback['recommendation']} — §13.8"
        )
        return round_summary

    def get_learning_curve(self) -> List[dict]:
        """Return the error progression across all rounds."""
        return [
            {
                "round": r["round"],
                "hypothesis": r["hypothesis_id"],
                "avg_error_pct": r["feedback"]["avg_error_pct"],
                "recommendation": r["feedback"]["recommendation"],
            }
            for r in self.rounds
        ]


# -- Demo: Run 3 rounds of closed-loop experiment tracking --

log_info("Demonstrating ExperimentTracker — Closed-Loop Learning (§13.8, pp.31-32)...")
print()

tracker = ExperimentTracker()

for round_data in MOCK_EXPERIMENT_ROUNDS:
    result = tracker.record_result(round_data)
print()

# Display learning curve
print("=" * 61)
print("  CLOSED-LOOP EXPERIMENT TRACKING (§13.8)")
print("  Prediction Error Reduction Over 3 Rounds")
print("=" * 61)

learning_curve = tracker.get_learning_curve()

print("+-------+------------------------+-----------+---------------+")
print("| Round | Hypothesis             | Avg Error | Status        |")
print("+-------+------------------------+-----------+---------------+")
for entry in learning_curve:
    err = entry["avg_error_pct"]
    status = entry["recommendation"]
    bar = "█" * int(err)
    print(f"| {entry['round']:<5d} | {entry['hypothesis']:<22s} | {err:>7.1f}%  | {status:<13s} |")
print("+-------+------------------------+-----------+---------------+")
print()

# Visual error reduction
print("  ERROR REDUCTION TRAJECTORY:")
for entry in learning_curve:
    err = entry["avg_error_pct"]
    bar = "█" * int(err * 2)
    spaces = " " * (30 - int(err * 2))
    print(f"    Round {entry['round']}: [{bar}{spaces}] {err:.1f}%")
print()

print("  DETAILED ROUND COMPARISON:")
print("  +------------+----------+---------+----------+")
print("  | Property   | Round 1  | Round 2 | Round 3  |")
print("  +------------+----------+---------+----------+")
properties = ["tg_celsius", "tensile_mpa", "elongation_pct"]
labels = ["Tg (C)", "Tensile (MPa)", "Elongation (%)"]
for prop, label in zip(properties, labels):
    vals = []
    for rd in MOCK_EXPERIMENT_ROUNDS:
        pred = rd["predicted"][prop]
        meas = rd["measured"][prop]
        err = abs(pred - meas) / meas * 100
        vals.append(f"{err:5.1f}%")
    print(f"  | {label:<10s} | {vals[0]:>7s} | {vals[1]:>7s} | {vals[2]:>7s}  |")
print("  +------------+----------+---------+----------+")
print()
log_success(
    f"ExperimentTracker demonstrated: error reduced from "
    f"{learning_curve[0]['avg_error_pct']:.0f}% -> "
    f"{learning_curve[1]['avg_error_pct']:.0f}% -> "
    f"{learning_curve[2]['avg_error_pct']:.0f}% (Level 4 learning)."
)

In [ ]:
# =============================================================================
# Cell 3e — Materials Science Case Study: Full Pipeline Orchestration
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.8 (pp.29-33) — Scientific Agent Orchestration
#
# Demonstrates the complete scientific discovery pipeline:
#   scan -> gap detection -> hypothesis generation -> experiment tracking
# Applied to the materials science case study from the chapter.
# =============================================================================

log_info("Running Materials Science Case Study — Full Pipeline (§13.8, pp.29-33)...")
print()

# ── Step 1: Literature Scan ──────────────────────────────────────────────
log_info("Step 1/4: Literature scanning...")
case_scanner = ProductionLiteratureScanner()
case_scan = case_scanner.search_all("high temperature polymer aerospace")
print()

# ── Step 2: Knowledge Gap Detection ──────────────────────────────────────
log_info("Step 2/4: Knowledge gap detection...")
case_gap_detector = KnowledgeGapDetector()
case_gaps = case_gap_detector.detect_gaps(case_scan["papers"])
print()

# ── Step 3: Hypothesis Generation ────────────────────────────────────────
log_info("Step 3/4: Hypothesis generation...")
case_hyp_gen = HypothesisGenerator()
case_hypotheses = case_hyp_gen.generate_hypotheses(case_gaps)
print()

# ── Step 4: Experiment Tracking ──────────────────────────────────────────
log_info("Step 4/4: Closed-loop experiment tracking...")
case_tracker = ExperimentTracker()
for rd in MOCK_EXPERIMENT_ROUNDS:
    case_tracker.record_result(rd)
case_curve = case_tracker.get_learning_curve()
print()

# ── Orchestration Summary ────────────────────────────────────────────────
print("=" * 61)
print("  MATERIALS SCIENCE CASE STUDY — FULL PIPELINE (§13.8)")
print("=" * 61)
print()
print("  PIPELINE EXECUTION SUMMARY:")
print("  +---+---------------------------+--------+-----------+")
print("  | # | Stage                     | Status | Output    |")
print("  +---+---------------------------+--------+-----------+")
print(f"  | 1 | Literature Scan            | [OK]   | {case_scan['total']:>4d} papers|")
print(f"  | 2 | Knowledge Gap Detection    | [OK]   | {len(case_gaps['gaps']):>4d} gaps  |")
print(f"  | 3 | Hypothesis Generation      | [OK]   | {len(case_hypotheses['hypotheses']):>4d} hyps  |")
print(f"  | 4 | Experiment Tracking        | [OK]   | {len(case_curve):>4d} rounds|")
print("  +---+---------------------------+--------+-----------+")
print()

# ── Timeline Comparison (from book §13.8, p.33) ─────────────────────────
print("  TIMELINE COMPARISON (§13.8, p.33):")
print("  +-----------------------------+-------------------+")
print("  | Approach                    | Time to Discovery |")
print("  +-----------------------------+-------------------+")
print("  | Traditional (manual)        | 9-12 months       |")
print("  | AI-Assisted (this agent)    | ~14 weeks         |")
print("  +-----------------------------+-------------------+")
print(f"  | Acceleration factor:        | ~3-4x faster      |")
print("  +-----------------------------+-------------------+")
print()

# ── Key Findings ─────────────────────────────────────────────────────────
final_error = case_curve[-1]["avg_error_pct"]
best_hyp = case_hypotheses["hypotheses"][0]
top_gap = case_gaps["gaps"][0]

print("  KEY FINDINGS:")
print(f"    - Top gap: {top_gap['description'][:50]}...")
print(f"    - Lead hypothesis: {best_hyp['id']} ({best_hyp['reasoning_type']})")
print(f"    - Final prediction error: {final_error:.1f}%")
print(f"    - Model status: {case_curve[-1]['recommendation']}")
print()

print("=" * 61)
print()
log_success(
    "Section 3 Complete — Scientific Discovery Agent fully demonstrated. "
    "All components from §13.5-§13.8 implemented and verified."
)

## Section 4: Cross-Domain Analysis & Summary (§13.9)

This final section compares the two agent architectures side by side
across multiple design dimensions, then summarizes the chapter's key
takeaways and pointers for further exploration.

In [ ]:
# =============================================================================
# Cell 4a — Cross-Domain Comparison Table
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §13.9 (pp.33-34) — Cross-Domain Analysis
#
# Compares the Healthcare Intelligence Agent and the Scientific
# Discovery Agent across 7 architectural dimensions.
# =============================================================================

log_info("Generating Cross-Domain Comparison Table (§13.9, pp.33-34)...")
print()

# Comparison data — each row is (dimension, healthcare, scientific)
comparison = [
    (
        "Correctness Model",
        "Bayesian posterior\nprobability",
        "Hypothesis consistency\n+ testability scores",
    ),
    (
        "Confidence\nCalibration",
        "Posterior distribution\nover differential dx",
        "Prediction error\nreduction over rounds",
    ),
    (
        "Data Access\nRegime",
        "Real-time EHR streams\n(FHIR-compliant)",
        "Batch literature\ncorpus (async fan-out)",
    ),
    (
        "Escalation\nStrategy",
        "MAP < 70 mmHg triggers\nhuman clinician review",
        "Novelty > 0.85 triggers\nmanual expert review",
    ),
    (
        "Explanation\nAudiences",
        "Clinician (technical)\n+ Patient (plain lang.)",
        "Research team\n(experimental protocol)",
    ),
    (
        "Learning\nBehavior",
        "Level 3: Contextual\nadaptation per patient",
        "Level 4: Closed-loop\nmodel refinement",
    ),
    (
        "Safety\nRequirements",
        "FDA/regulatory audit\ntrail mandatory",
        "Reproducibility log\n+ version control",
    ),
]

# Calculate column widths
col_w = [20, 24, 24]
total_w = sum(col_w) + 4 + 2  # 4 separators + 2 edges

print("=" * (total_w + 4))
print("  CROSS-DOMAIN COMPARISON: Healthcare vs Scientific Agent (§13.9)")
print("=" * (total_w + 4))
print()

# Header
hdr_fmt = f"| {{:<{col_w[0]}}} | {{:<{col_w[1]}}} | {{:<{col_w[2]}}} |"
sep_line = f"+{'-' * (col_w[0]+2)}+{'-' * (col_w[1]+2)}+{'-' * (col_w[2]+2)}+"

print(sep_line)
print(hdr_fmt.format("Dimension", "Healthcare Agent", "Scientific Agent"))
print(hdr_fmt.format("", "(§13.1-§13.4)", "(§13.5-§13.8)"))
print(sep_line)

# Rows — handle multi-line cells
for dim, health, science in comparison:
    dim_lines = dim.split("\n")
    health_lines = health.split("\n")
    science_lines = science.split("\n")
    max_lines = max(len(dim_lines), len(health_lines), len(science_lines))

    for row_i in range(max_lines):
        d = dim_lines[row_i] if row_i < len(dim_lines) else ""
        h = health_lines[row_i] if row_i < len(health_lines) else ""
        s = science_lines[row_i] if row_i < len(science_lines) else ""
        print(hdr_fmt.format(d, h, s))
    print(sep_line)

print()

# Additional architectural observations
print("  KEY ARCHITECTURAL OBSERVATIONS (§13.9):")
print()
observations = [
    "Both agents use a coordinator pattern — one master class orchestrating",
    "  multiple specialist sub-agents through a defined workflow.",
    "",
    "The Healthcare Agent prioritizes SAFETY (audit trails, escalation)",
    "  while the Scientific Agent prioritizes DISCOVERY (gap detection,",
    "  hypothesis novelty scoring).",
    "",
    "Simulation Mode enables both agents to run identically in mock and",
    "  live modes — the @graceful_fallback decorator ensures seamless",
    "  degradation from live APIs to deterministic mock responses.",
    "",
    "The closed-loop learning pattern (§13.8) is applicable to both",
    "  domains: healthcare agents can refine diagnostic accuracy over",
    "  patient cohorts just as scientific agents refine predictions",
    "  over experimental rounds.",
]
for line in observations:
    print(f"    {line}")
print()
print("=" * (total_w + 4))
print()
log_success("Cross-domain comparison table generated (§13.9).")

In [ ]:
# =============================================================================
# Cell 4b — Chapter Summary and Next Steps
# Chapter 13: Healthcare and Scientific Agents
# Book: 30 Agents Every AI Engineer Must Build
# Author: Imran Ahmad
# Section Ref: §Summary (pp.34-35)
# =============================================================================

print()
print("=" * 61)
print("  CHAPTER 13 SUMMARY")
print("  Healthcare and Scientific Agents")
print("  Book: 30 Agents Every AI Engineer Must Build")
print("  Author: Imran Ahmad")
print("=" * 61)
print()
print("  KEY TAKEAWAYS:")
print()
print("  1. BAYESIAN REASONING provides a principled framework for")
print("     updating diagnostic beliefs as new evidence arrives.")
print("     (§13.1 — update_belief function)")
print()
print("  2. MULTI-SOURCE KNOWLEDGE INTEGRATION with provenance")
print("     tracking ensures that clinical decisions are traceable")
print("     back to their evidence sources. (§13.1 — ClinicalKB)")
print()
print("  3. FHIR NORMALIZATION enables interoperability across")
print("     heterogeneous EHR systems — a critical production")
print("     requirement. (§13.2 — FHIRNormalizationLayer)")
print()
print("  4. MULTI-SPECIALIST COORDINATION through a central")
print("     DiagnosticCoordinator allows modular agent design")
print("     where each specialist can be upgraded independently.")
print("     (§13.3 — DiagnosticCoordinator)")
print()
print("  5. DUAL-AUDIENCE EXPLANATIONS bridge the gap between")
print("     clinical precision and patient comprehension.")
print("     (§13.3 — ClinicalExplainer)")
print()
print("  6. ASYNC LITERATURE SCANNING with caching and dedup")
print("     scales to production workloads across multiple")
print("     academic databases. (§13.5 — LiteratureScanner)")
print()
print("  7. KNOWLEDGE GAP DETECTION uses three complementary")
print("     strategies to surface research opportunities that")
print("     human researchers might miss. (§13.6 — GapDetector)")
print()
print("  8. CLOSED-LOOP LEARNING enables Level 4 autonomy —")
print("     the agent refines its own predictions from")
print("     experimental outcomes. (§13.8 — ExperimentTracker)")
print()
print("-" * 61)
print("  SIMULATION MODE STATISTICS:")

# Gather stats
if SIMULATION_MODE:
    mock_calls = llm.get_call_count()
    print(f"    MockLLM total invocations: {mock_calls}")
    print(f"    Context types exercised:   6 of 6")
    print(f"    Decorated methods:         8 of 8")
    print(f"    Mock datasets used:        9 of 9")
    print(f"    Errors encountered:        0")
else:
    print("    Running in LIVE mode — statistics from real API.")

print()
print("-" * 61)
print("  NEXT STEPS FOR THE READER:")
print()
print("  - Modify MOCK_PATIENT_VITALS to test different clinical")
print("    scenarios (e.g., change temperature, MAP, lactate)")
print("  - Adjust the ESCALATION_THRESHOLD (0.15) in")
print("    DiagnosticCoordinator and observe behavior changes")
print("  - Add a new diagnosis to MOCK_DIAGNOSES and update")
print("    the prior belief and likelihood arrays")
print("  - Extend the paper corpus with new entries and run")
print("    the gap detector to see how clusters shift")
print("  - Add a 4th experimental round to MOCK_EXPERIMENT_ROUNDS")
print("    and verify the learning curve continues to converge")
print()
print("=" * 61)
print("  End of Chapter 13 Notebook")
print("  Author: Imran Ahmad")
print("  Book: 30 Agents Every AI Engineer Must Build")
print("=" * 61)
print()
log_success("Chapter 13 notebook execution complete. All sections verified.")